In [1]:
# 0.0 - mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0 - Load Frozen Artefacts + Cross-File Integrity Checks
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0.1 Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

B1_RESULTS_ROOT = (
    RESULTS_ROOT
    / "B1_experiments"
)


# ------------------------------------------------------------
# 0.2 Frozen input files
# ------------------------------------------------------------

B0_PER_IMAGE_FILE = (
    RESULTS_ROOT
    / "B0_dev100"
    / "B0_dev100_per_image.csv"
)

C_PER_IMAGE_FILE = (
    B1_RESULTS_ROOT
    / "Group_C"
    / "B1_Group_C_E_candidates_per_image.csv"
)

D_PER_IMAGE_FILE = (
    B1_RESULTS_ROOT
    / "Group_D"
    / "B1_Group_D_E_candidates_per_image.csv"
)

E2_PER_IMAGE_FILE = (
    B1_RESULTS_ROOT
    / "Group_E"
    / "E2_GT_complementarity"
    / "E2_all_pairs_per_image.csv"
)

E3_FUSION_FILE = (
    B1_RESULTS_ROOT
    / "Group_E"
    / "E3_semantic_integration"
    / "fusion_generation"
    / "E3_fusion_per_image.csv"
)

E3_EVAL_FILE = (
    B1_RESULTS_ROOT
    / "Group_E"
    / "E3_semantic_integration"
    / "gt_evaluation"
    / "E3_fusion_eval_per_image.csv"
)

COUNT_COMPARISON_FILE = (
    B1_RESULTS_ROOT
    / "End_to_End_Checkpoint"
    / "tables"
    / "count_comparison_per_image.csv"
)

COUNT_CHANGE_FILE = (
    B1_RESULTS_ROOT
    / "End_to_End_Checkpoint"
    / "tables"
    / "count_error_change_per_image_vs_b0.csv"
)


# ------------------------------------------------------------
# 0.3 New diagnostic output directory
# ------------------------------------------------------------

DIAG_RESULTS_DIR = (
    B1_RESULTS_ROOT
    / "Sparse_Scene_Failure_Diagnostic"
)

DIAG_TABLES_DIR = (
    DIAG_RESULTS_DIR
    / "tables"
)

DIAG_METADATA_DIR = (
    DIAG_RESULTS_DIR
    / "metadata"
)

for path in [
    DIAG_RESULTS_DIR,
    DIAG_TABLES_DIR,
    DIAG_METADATA_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 0.4 Check that all eight frozen inputs exist
# ------------------------------------------------------------

INPUT_FILES = {
    "B0":
        B0_PER_IMAGE_FILE,

    "C":
        C_PER_IMAGE_FILE,

    "D":
        D_PER_IMAGE_FILE,

    "E2":
        E2_PER_IMAGE_FILE,

    "E3_fusion":
        E3_FUSION_FILE,

    "E3_eval":
        E3_EVAL_FILE,

    "count_comparison":
        COUNT_COMPARISON_FILE,

    "count_change":
        COUNT_CHANGE_FILE,
}


missing_files = [
    name
    for name, path
    in INPUT_FILES.items()
    if not path.exists()
]

assert (
    len(
        missing_files
    )
    == 0
), (
    "Missing frozen input files: "
    f"{missing_files}"
)


# ------------------------------------------------------------
# 0.5 Load all frozen tables
# ------------------------------------------------------------

b0_df = pd.read_csv(
    B0_PER_IMAGE_FILE
)

c_df = pd.read_csv(
    C_PER_IMAGE_FILE
)

d_df = pd.read_csv(
    D_PER_IMAGE_FILE
)

e2_df = pd.read_csv(
    E2_PER_IMAGE_FILE
)

e3_fusion_df = pd.read_csv(
    E3_FUSION_FILE
)

e3_eval_df = pd.read_csv(
    E3_EVAL_FILE
)

count_comparison_df = pd.read_csv(
    COUNT_COMPARISON_FILE
)

count_change_df = pd.read_csv(
    COUNT_CHANGE_FILE
)


# ------------------------------------------------------------
# 0.6 Expected frozen structure
# ------------------------------------------------------------

DENSITY_ORDER = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]

SPARSE_STRATA = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
]

DENSE_STRATA = [
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]

EXPECTED_C_VARIANTS = [
    "C1",
    "C2b",
    "C2c",
]

EXPECTED_D_VARIANTS = [
    "D1",
    "D4",
    "D5-768",
    "D6B-T-native",
    "D6B9-strict",
]

EXPECTED_GUIDED_SYSTEMS = [
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
]

EXPECTED_COUNT_SYSTEMS = [
    "B0_unguided",
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
]


# ------------------------------------------------------------
# 0.7 Row-count checks
# ------------------------------------------------------------

EXPECTED_ROWS = {
    "B0":
        100,

    "C":
        300,       # 3 variants x 100 images

    "D":
        500,       # 5 variants x 100 images

    "E2":
        1500,      # 3 C x 5 D x 100 images

    "E3_fusion":
        22500,

    "E3_eval":
        22500,

    "count_comparison":
        400,       # 4 systems x 100 images

    "count_change":
        300,       # 3 guided systems x 100 images
}


observed_rows = {
    "B0":
        len(b0_df),

    "C":
        len(c_df),

    "D":
        len(d_df),

    "E2":
        len(e2_df),

    "E3_fusion":
        len(e3_fusion_df),

    "E3_eval":
        len(e3_eval_df),

    "count_comparison":
        len(count_comparison_df),

    "count_change":
        len(count_change_df),
}


for name in EXPECTED_ROWS:

    assert (
        observed_rows[name]
        == EXPECTED_ROWS[name]
    ), (
        f"{name}: observed "
        f"{observed_rows[name]} rows, "
        f"expected {EXPECTED_ROWS[name]}"
    )


# ------------------------------------------------------------
# 0.8 Candidate/system identity checks
# ------------------------------------------------------------

observed_c_variants = sorted(
    c_df[
        "candidate_id"
    ]
    .unique()
    .tolist()
)

observed_d_variants = sorted(
    d_df[
        "candidate_id"
    ]
    .unique()
    .tolist()
)

observed_guided_systems = sorted(
    count_change_df[
        "checkpoint_id"
    ]
    .unique()
    .tolist()
)

observed_count_systems = (
    count_comparison_df[
        "system_id"
    ]
    .drop_duplicates()
    .tolist()
)


assert (
    observed_c_variants
    == sorted(
        EXPECTED_C_VARIANTS
    )
)

assert (
    observed_d_variants
    == sorted(
        EXPECTED_D_VARIANTS
    )
)

assert (
    observed_guided_systems
    == sorted(
        EXPECTED_GUIDED_SYSTEMS
    )
)

assert (
    observed_count_systems
    == EXPECTED_COUNT_SYSTEMS
)


# ------------------------------------------------------------
# 0.9 Build authoritative dev100 image mapping from B0
# ------------------------------------------------------------

dev100_df = (
    b0_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .copy()
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        dev100_df
    )
    == 100
)

assert (
    dev100_df[
        "dataset_index"
    ]
    .is_unique
)

assert (
    dev100_df[
        "image_id"
    ]
    .is_unique
)


# ------------------------------------------------------------
# 0.10 Helper: validate image-level identity against B0
# ------------------------------------------------------------

def validate_image_mapping(
    df,
    name,
    include_category=True,
    include_gt=True,
):

    cols = [
        "dataset_index",
        "image_id",
        "density_stratum",
    ]

    if (
        include_category
        and
        "category" in df.columns
    ):
        cols.append(
            "category"
        )

    if (
        include_gt
        and
        "gt_count" in df.columns
    ):
        cols.append(
            "gt_count"
        )


    unique_df = (
        df[
            cols
        ]
        .drop_duplicates()
    )


    expected_cols = [
        col
        for col
        in cols
        if col
        in dev100_df.columns
    ]


    expected_df = (
        dev100_df[
            expected_cols
        ]
        .drop_duplicates()
    )


    merged = (
        unique_df
        .merge(
            expected_df,
            on=expected_cols,
            how="outer",
            indicator=True,
        )
    )


    assert (
        merged[
            "_merge"
        ]
        .eq(
            "both"
        )
        .all()
    ), (
        f"{name}: image / GT / density "
        "mapping differs from frozen B0."
    )


# ------------------------------------------------------------
# 0.11 Cross-file image mapping checks
# ------------------------------------------------------------

validate_image_mapping(
    c_df,
    "C",
)

validate_image_mapping(
    d_df,
    "D",
)

validate_image_mapping(
    e2_df,
    "E2",
    include_category=False,
)

validate_image_mapping(
    e3_eval_df,
    "E3_eval",
)

validate_image_mapping(
    count_comparison_df,
    "count_comparison",
)

validate_image_mapping(
    count_change_df,
    "count_change",
)


# E3 fusion-generation table does not contain
# density_stratum or GT count, so validate image identity only.

e3_fusion_images = (
    e3_fusion_df[
        [
            "dataset_index",
            "image_id",
            "category",
        ]
    ]
    .drop_duplicates()
)

expected_e3_images = (
    dev100_df[
        [
            "dataset_index",
            "image_id",
            "category",
        ]
    ]
)

e3_mapping_check = (
    e3_fusion_images
    .merge(
        expected_e3_images,
        on=[
            "dataset_index",
            "image_id",
            "category",
        ],
        how="outer",
        indicator=True,
    )
)

assert (
    e3_mapping_check[
        "_merge"
    ]
    .eq(
        "both"
    )
    .all()
)


# ------------------------------------------------------------
# 0.12 Validate density-stratum image counts
# ------------------------------------------------------------

density_counts = (
    dev100_df[
        "density_stratum"
    ]
    .value_counts()
    .reindex(
        DENSITY_ORDER
    )
)


expected_density_counts = pd.Series(
    {
        "S1_0_20":
            20,

        "S2_20_40":
            20,

        "S3_40_60":
            20,

        "S4_60_80":
            20,

        "S5_80_95":
            10,

        "S6_95_100":
            10,
    }
)


assert (
    density_counts
    .astype(int)
    .equals(
        expected_density_counts
    )
)


# ------------------------------------------------------------
# 0.13 Validate sparse diagnostic subset
# ------------------------------------------------------------

sparse_dev_df = (
    dev100_df[
        dev100_df[
            "density_stratum"
        ].isin(
            SPARSE_STRATA
        )
    ]
    .copy()
)


assert (
    len(
        sparse_dev_df
    )
    == 60
)


assert (
    sparse_dev_df[
        "density_stratum"
    ]
    .value_counts()
    .reindex(
        SPARSE_STRATA
    )
    .tolist()
    == [
        20,
        20,
        20,
    ]
)


# ------------------------------------------------------------
# 0.14 Per-variant/per-system completeness checks
# ------------------------------------------------------------

c_image_counts = (
    c_df
    .groupby(
        "candidate_id"
    )[
        "dataset_index"
    ]
    .nunique()
)

d_image_counts = (
    d_df
    .groupby(
        "candidate_id"
    )[
        "dataset_index"
    ]
    .nunique()
)

e2_pair_counts = (
    e2_df
    .groupby(
        [
            "c_variant",
            "d_variant",
        ]
    )[
        "dataset_index"
    ]
    .nunique()
)

e3_config_counts = (
    e3_eval_df
    .groupby(
        [
            "c_variant",
            "d_variant",
            "tau",
            "fusion_family",
        ]
    )[
        "dataset_index"
    ]
    .nunique()
)

count_system_counts = (
    count_comparison_df
    .groupby(
        "system_id"
    )[
        "dataset_index"
    ]
    .nunique()
)

guided_system_counts = (
    count_change_df
    .groupby(
        "checkpoint_id"
    )[
        "dataset_index"
    ]
    .nunique()
)


assert (
    c_image_counts
    == 100
).all()

assert (
    d_image_counts
    == 100
).all()

assert (
    e2_pair_counts
    == 100
).all()

assert (
    e3_config_counts
    == 100
).all()

assert (
    count_system_counts
    == 100
).all()

assert (
    guided_system_counts
    == 100
).all()


# ------------------------------------------------------------
# 0.15 Validate E3 generation and E3 evaluation identities
# ------------------------------------------------------------

e3_key_cols = [
    "dataset_index",
    "image_id",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
    "pred_count",
]


e3_identity_check = (
    e3_fusion_df[
        e3_key_cols
    ]
    .merge(
        e3_eval_df[
            e3_key_cols
        ],
        on=e3_key_cols,
        how="outer",
        indicator=True,
    )
)


assert (
    len(
        e3_identity_check
    )
    == 22500
)

assert (
    e3_identity_check[
        "_merge"
    ]
    .eq(
        "both"
    )
    .all()
)


# ------------------------------------------------------------
# 0.16 Basic numeric sanity checks
# ------------------------------------------------------------

for df, name in [
    (
        c_df,
        "C",
    ),
    (
        d_df,
        "D",
    ),
    (
        e2_df,
        "E2",
    ),
    (
        e3_eval_df,
        "E3_eval",
    ),
    (
        count_comparison_df,
        "count_comparison",
    ),
    (
        count_change_df,
        "count_change",
    ),
]:

    if "gt_count" in df.columns:

        assert (
            df[
                "gt_count"
            ]
            > 0
        ).all(), (
            f"{name}: non-positive GT count found."
        )


for df, column, name in [
    (
        c_df,
        "pred_count",
        "C",
    ),
    (
        d_df,
        "pred_count",
        "D",
    ),
    (
        e3_fusion_df,
        "pred_count",
        "E3_fusion",
    ),
    (
        e3_eval_df,
        "pred_count",
        "E3_eval",
    ),
    (
        count_comparison_df,
        "pred_count",
        "count_comparison",
    ),
]:

    assert (
        df[
            column
        ]
        >= 0
    ).all(), (
        f"{name}: negative prediction count found."
    )


# ------------------------------------------------------------
# 0.17 No test-set inference is performed here
# ------------------------------------------------------------

DIAGNOSTIC_SCOPE = {
    "dataset":
        "FSC-147",

    "subset":
        "frozen dev100",

    "num_images":
        100,

    "primary_sparse_subset":
        "S1-S3",

    "num_sparse_images":
        60,

    "new_model_inference":
        False,

    "threshold_tuning":
        False,

    "test_set_accessed":
        False,
}


# ------------------------------------------------------------
# 0.18 Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 0 - FROZEN INPUT LOAD + CROSS-FILE INTEGRITY")
print("=" * 110)

print(
    "All 8 frozen input files     : PASS"
)

print(
    "Expected row counts          : PASS"
)

print(
    "C variants                   :",
    observed_c_variants,
)

print(
    "D variants                   :",
    observed_d_variants,
)

print(
    "Guided systems               :",
    observed_guided_systems,
)

print(
    "Count systems                :",
    observed_count_systems,
)

print(
    "dev100 unique images         :",
    len(
        dev100_df
    ),
)

print(
    "Sparse S1-S3 images          :",
    len(
        sparse_dev_df
    ),
)

print(
    "Cross-file image mapping     : PASS"
)

print(
    "Density-stratum mapping      : PASS"
)

print(
    "Variant completeness         : PASS"
)

print(
    "E3 generation/eval identity : PASS"
)

print(
    "Basic numeric sanity         : PASS"
)

print(
    "New model inference          : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nFrozen input table shapes:"
)

shape_df = pd.DataFrame(
    {
        "table":
            [
                "B0",
                "C",
                "D",
                "E2",
                "E3_fusion",
                "E3_eval",
                "count_comparison",
                "count_change",
            ],

        "rows":
            [
                len(
                    b0_df
                ),

                len(
                    c_df
                ),

                len(
                    d_df
                ),

                len(
                    e2_df
                ),

                len(
                    e3_fusion_df
                ),

                len(
                    e3_eval_df
                ),

                len(
                    count_comparison_df
                ),

                len(
                    count_change_df
                ),
            ],

        "columns":
            [
                len(
                    b0_df.columns
                ),

                len(
                    c_df.columns
                ),

                len(
                    d_df.columns
                ),

                len(
                    e2_df.columns
                ),

                len(
                    e3_fusion_df.columns
                ),

                len(
                    e3_eval_df.columns
                ),

                len(
                    count_comparison_df.columns
                ),

                len(
                    count_change_df.columns
                ),
            ],
    }
)

display(
    shape_df
)


print(
    "\nFrozen dev100 density distribution:"
)

display(
    density_counts
    .rename(
        "num_images"
    )
    .to_frame()
)


print(
    "\nDiagnostic output directory:"
)

print(
    DIAG_RESULTS_DIR
)

print("=" * 110)

STEP 0 - FROZEN INPUT LOAD + CROSS-FILE INTEGRITY
All 8 frozen input files     : PASS
Expected row counts          : PASS
C variants                   : ['C1', 'C2b', 'C2c']
D variants                   : ['D1', 'D4', 'D5-768', 'D6B-T-native', 'D6B9-strict']
Guided systems               : ['dense_recovery', 'overall_joint', 'sparse_joint']
Count systems                : ['B0_unguided', 'overall_joint', 'sparse_joint', 'dense_recovery']
dev100 unique images         : 100
Sparse S1-S3 images          : 60
Cross-file image mapping     : PASS
Density-stratum mapping      : PASS
Variant completeness         : PASS
E3 generation/eval identity : PASS
Basic numeric sanity         : PASS
New model inference          : NO
Threshold tuning             : NO
Test set accessed            : NO

Frozen input table shapes:


,table,rows,columns
0,B0,100,13
1,C,300,27
2,D,500,48
3,E2,1500,20
4,E3_fusion,22500,9
5,E3_eval,22500,24
6,count_comparison,400,12
7,count_change,300,21



Frozen dev100 density distribution:


,num_images
density_stratum,
S1_0_20,20
S2_20_40,20
S3_40_60,20
S4_60_80,20
S5_80_95,10
S6_95_100,10



Diagnostic output directory:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic


In [3]:
# ============================================================
# Step 1 - Q1: C/D Point-Burden Decomposition in S1-S3
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1.1 Diagnostic output files
# ------------------------------------------------------------

Q1_BY_STRATUM_FILE = (
    DIAG_TABLES_DIR
    / "Q1_CD_point_burden_by_sparse_stratum.csv"
)

Q1_OVERALL_SPARSE_FILE = (
    DIAG_TABLES_DIR
    / "Q1_CD_point_burden_overall_S1_S3.csv"
)

Q1_PER_IMAGE_FILE = (
    DIAG_TABLES_DIR
    / "Q1_CD_point_burden_per_image_S1_S3.csv"
)


# ------------------------------------------------------------
# 1.2 Required columns
# ------------------------------------------------------------

COMMON_REQUIRED = {
    "candidate_id",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "pred_count",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
}


for df, name in [
    (c_df, "C"),
    (d_df, "D"),
]:

    missing = (
        COMMON_REQUIRED
        - set(
            df.columns
        )
    )

    assert (
        len(
            missing
        )
        == 0
    ), (
        f"{name} missing required columns: "
        f"{sorted(missing)}"
    )


# ------------------------------------------------------------
# 1.3 Restrict to frozen sparse subset S1-S3
# ------------------------------------------------------------

c_sparse_df = (
    c_df[
        c_df[
            "density_stratum"
        ].isin(
            SPARSE_STRATA
        )
    ]
    .copy()
)

d_sparse_df = (
    d_df[
        d_df[
            "density_stratum"
        ].isin(
            SPARSE_STRATA
        )
    ]
    .copy()
)


assert (
    len(
        c_sparse_df
    )
    == 3 * 60
)

assert (
    len(
        d_sparse_df
    )
    == 5 * 60
)


# ------------------------------------------------------------
# 1.4 Add harmonised burden measures
#
# Do not rely on pre-existing ratio columns because
# C and D may not use identical schemas.
# ------------------------------------------------------------

def add_q1_metrics(
    df,
    branch,
):

    out = df.copy()

    out[
        "branch"
    ] = branch


    # --------------------------------------------------------
    # Point burden relative to GT count
    # --------------------------------------------------------

    out[
        "pred_gt_ratio_q1"
    ] = (
        out[
            "pred_count"
        ]
        /
        out[
            "gt_count"
        ]
    )


    # --------------------------------------------------------
    # Signed point-count excess
    #
    # > 0 : more predicted points than GT objects
    # < 0 : fewer predicted points than GT objects
    # --------------------------------------------------------

    out[
        "point_count_excess"
    ] = (
        out[
            "pred_count"
        ]
        -
        out[
            "gt_count"
        ]
    )


    # --------------------------------------------------------
    # Relative point-count excess
    # --------------------------------------------------------

    out[
        "relative_point_excess"
    ] = (
        out[
            "point_count_excess"
        ]
        /
        out[
            "gt_count"
        ]
    )


    # --------------------------------------------------------
    # FP burden per matched GT point
    #
    # This is a diagnostic ratio, not a standard metric.
    # If TP=0 and FP>0 -> infinity.
    # --------------------------------------------------------

    out[
        "fp_per_tp"
    ] = np.where(
        out[
            "tp"
        ] > 0,

        out[
            "fp"
        ]
        /
        out[
            "tp"
        ],

        np.where(
            out[
                "fp"
            ] > 0,
            np.inf,
            0.0,
        ),
    )


    # --------------------------------------------------------
    # Simple count-direction label
    # --------------------------------------------------------

    out[
        "count_direction"
    ] = np.select(
        [
            out[
                "pred_count"
            ]
            > out[
                "gt_count"
            ],

            out[
                "pred_count"
            ]
            < out[
                "gt_count"
            ],
        ],
        [
            "over",
            "under",
        ],
        default="exact",
    )


    return out


c_sparse_q1_df = add_q1_metrics(
    c_sparse_df,
    "C",
)

d_sparse_q1_df = add_q1_metrics(
    d_sparse_df,
    "D",
)


q1_per_image_df = (
    pd.concat(
        [
            c_sparse_q1_df,
            d_sparse_q1_df,
        ],
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# 1.5 Strong integrity checks
# ------------------------------------------------------------

assert (
    len(
        q1_per_image_df
    )
    == 480
)


assert (
    q1_per_image_df[
        "gt_count"
    ]
    > 0
).all()


# Point-matching identities

assert (
    q1_per_image_df[
        "tp"
    ]
    +
    q1_per_image_df[
        "fp"
    ]
    ==
    q1_per_image_df[
        "pred_count"
    ]
).all()


assert (
    q1_per_image_df[
        "tp"
    ]
    +
    q1_per_image_df[
        "fn"
    ]
    ==
    q1_per_image_df[
        "gt_count"
    ]
).all()


# Recompute precision/recall from TP/FP/FN and verify

precision_check = np.where(
    (
        q1_per_image_df[
            "tp"
        ]
        +
        q1_per_image_df[
            "fp"
        ]
    )
    > 0,

    q1_per_image_df[
        "tp"
    ]
    /
    (
        q1_per_image_df[
            "tp"
        ]
        +
        q1_per_image_df[
            "fp"
        ]
    ),

    0.0,
)


recall_check = (
    q1_per_image_df[
        "tp"
    ]
    /
    (
        q1_per_image_df[
            "tp"
        ]
        +
        q1_per_image_df[
            "fn"
        ]
    )
)


assert np.allclose(
    precision_check,
    q1_per_image_df[
        "precision"
    ],
)

assert np.allclose(
    recall_check,
    q1_per_image_df[
        "recall"
    ],
)


# ------------------------------------------------------------
# 1.6 Aggregation helper
# ------------------------------------------------------------

def summarise_q1(
    df,
    group_cols,
):

    records = []


    for keys, group in df.groupby(
        group_cols,
        sort=False,
    ):

        if not isinstance(
            keys,
            tuple,
        ):
            keys = (
                keys,
            )


        record = dict(
            zip(
                group_cols,
                keys,
            )
        )


        num_images = int(
            len(
                group
            )
        )

        gt_sum = int(
            group[
                "gt_count"
            ].sum()
        )

        pred_sum = int(
            group[
                "pred_count"
            ].sum()
        )

        tp_sum = int(
            group[
                "tp"
            ].sum()
        )

        fp_sum = int(
            group[
                "fp"
            ].sum()
        )

        fn_sum = int(
            group[
                "fn"
            ].sum()
        )


        pooled_precision = (
            tp_sum
            /
            (
                tp_sum
                +
                fp_sum
            )
            if (
                tp_sum
                +
                fp_sum
            )
            > 0
            else 0.0
        )


        pooled_recall = (
            tp_sum
            /
            (
                tp_sum
                +
                fn_sum
            )
            if (
                tp_sum
                +
                fn_sum
            )
            > 0
            else 0.0
        )


        pooled_f1 = (
            2.0
            * pooled_precision
            * pooled_recall
            /
            (
                pooled_precision
                +
                pooled_recall
            )
            if (
                pooled_precision
                +
                pooled_recall
            )
            > 0
            else 0.0
        )


        fp_per_tp_pooled = (
            fp_sum
            /
            tp_sum
            if tp_sum > 0
            else np.inf
        )


        record.update(
            {
                "num_images":
                    num_images,

                # --------------------------------------------
                # GT / prediction burden
                # --------------------------------------------

                "mean_gt_count":
                    float(
                        group[
                            "gt_count"
                        ].mean()
                    ),

                "median_gt_count":
                    float(
                        group[
                            "gt_count"
                        ].median()
                    ),

                "mean_pred_count":
                    float(
                        group[
                            "pred_count"
                        ].mean()
                    ),

                "median_pred_count":
                    float(
                        group[
                            "pred_count"
                        ].median()
                    ),

                "mean_pred_gt_ratio":
                    float(
                        group[
                            "pred_gt_ratio_q1"
                        ].mean()
                    ),

                "median_pred_gt_ratio":
                    float(
                        group[
                            "pred_gt_ratio_q1"
                        ].median()
                    ),

                "aggregate_pred_gt_ratio":
                    float(
                        pred_sum
                        /
                        gt_sum
                    ),

                "mean_point_count_excess":
                    float(
                        group[
                            "point_count_excess"
                        ].mean()
                    ),

                "median_point_count_excess":
                    float(
                        group[
                            "point_count_excess"
                        ].median()
                    ),

                # --------------------------------------------
                # Direction of count burden
                # --------------------------------------------

                "over_images":
                    int(
                        (
                            group[
                                "count_direction"
                            ]
                            == "over"
                        ).sum()
                    ),

                "exact_images":
                    int(
                        (
                            group[
                                "count_direction"
                            ]
                            == "exact"
                        ).sum()
                    ),

                "under_images":
                    int(
                        (
                            group[
                                "count_direction"
                            ]
                            == "under"
                        ).sum()
                    ),

                "over_image_rate":
                    float(
                        (
                            group[
                                "count_direction"
                            ]
                            == "over"
                        ).mean()
                    ),

                # --------------------------------------------
                # Point matching totals
                # --------------------------------------------

                "total_gt":
                    gt_sum,

                "total_pred":
                    pred_sum,

                "total_tp":
                    tp_sum,

                "total_fp":
                    fp_sum,

                "total_fn":
                    fn_sum,

                # --------------------------------------------
                # Pooled localisation metrics
                # --------------------------------------------

                "pooled_precision":
                    float(
                        pooled_precision
                    ),

                "pooled_recall":
                    float(
                        pooled_recall
                    ),

                "pooled_f1":
                    float(
                        pooled_f1
                    ),

                "pooled_fp_per_tp":
                    float(
                        fp_per_tp_pooled
                    ),

                # --------------------------------------------
                # Mean image-level metrics
                # --------------------------------------------

                "mean_image_precision":
                    float(
                        group[
                            "precision"
                        ].mean()
                    ),

                "mean_image_recall":
                    float(
                        group[
                            "recall"
                        ].mean()
                    ),

                "mean_image_f1":
                    float(
                        group[
                            "f1"
                        ].mean()
                    ),
            }
        )


        records.append(
            record
        )


    return pd.DataFrame(
        records
    )


# ------------------------------------------------------------
# 1.7 S1 / S2 / S3 summaries
# ------------------------------------------------------------

q1_by_stratum_df = (
    summarise_q1(
        q1_per_image_df,
        [
            "branch",
            "candidate_id",
            "density_stratum",
        ],
    )
)


# Put strata in frozen order

q1_by_stratum_df[
    "density_stratum"
] = pd.Categorical(
    q1_by_stratum_df[
        "density_stratum"
    ],
    categories=SPARSE_STRATA,
    ordered=True,
)


q1_by_stratum_df = (
    q1_by_stratum_df
    .sort_values(
        [
            "branch",
            "candidate_id",
            "density_stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        q1_by_stratum_df
    )
    == (
        3 * 3
        +
        5 * 3
    )
)


# ------------------------------------------------------------
# 1.8 Combined S1-S3 summary
# ------------------------------------------------------------

q1_overall_sparse_df = (
    summarise_q1(
        q1_per_image_df,
        [
            "branch",
            "candidate_id",
        ],
    )
    .sort_values(
        [
            "branch",
            "aggregate_pred_gt_ratio",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        q1_overall_sparse_df
    )
    == 8
)


# ------------------------------------------------------------
# 1.9 Save frozen Q1 diagnostic outputs
# ------------------------------------------------------------

q1_per_image_df.to_csv(
    Q1_PER_IMAGE_FILE,
    index=False,
)

q1_by_stratum_df.to_csv(
    Q1_BY_STRATUM_FILE,
    index=False,
)

q1_overall_sparse_df.to_csv(
    Q1_OVERALL_SPARSE_FILE,
    index=False,
)


# ------------------------------------------------------------
# 1.10 Display compact burden tables
# ------------------------------------------------------------

DISPLAY_COLS = [
    "branch",
    "candidate_id",
    "density_stratum",
    "mean_gt_count",
    "mean_pred_count",
    "aggregate_pred_gt_ratio",
    "total_tp",
    "total_fp",
    "total_fn",
    "pooled_precision",
    "pooled_recall",
    "pooled_f1",
    "pooled_fp_per_tp",
    "over_image_rate",
]


display_by_stratum_df = (
    q1_by_stratum_df[
        DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "mean_gt_count",
    "mean_pred_count",
    "aggregate_pred_gt_ratio",
    "pooled_precision",
    "pooled_recall",
    "pooled_f1",
    "pooled_fp_per_tp",
    "over_image_rate",
]:

    display_by_stratum_df[
        col
    ] = (
        display_by_stratum_df[
            col
        ]
        .round(
            4
        )
    )


OVERALL_DISPLAY_COLS = [
    "branch",
    "candidate_id",
    "mean_gt_count",
    "mean_pred_count",
    "aggregate_pred_gt_ratio",
    "total_gt",
    "total_pred",
    "total_tp",
    "total_fp",
    "total_fn",
    "pooled_precision",
    "pooled_recall",
    "pooled_f1",
    "pooled_fp_per_tp",
    "over_image_rate",
]


display_overall_df = (
    q1_overall_sparse_df[
        OVERALL_DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "mean_gt_count",
    "mean_pred_count",
    "aggregate_pred_gt_ratio",
    "pooled_precision",
    "pooled_recall",
    "pooled_f1",
    "pooled_fp_per_tp",
    "over_image_rate",
]:

    display_overall_df[
        col
    ] = (
        display_overall_df[
            col
        ]
        .round(
            4
        )
    )


# ------------------------------------------------------------
# 1.11 Useful pivots for visual comparison
# ------------------------------------------------------------

pred_gt_ratio_pivot = (
    q1_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="aggregate_pred_gt_ratio",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


precision_pivot = (
    q1_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="pooled_precision",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


recall_pivot = (
    q1_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="pooled_recall",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


fp_per_tp_pivot = (
    q1_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="pooled_fp_per_tp",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


# ------------------------------------------------------------
# 1.12 Confirmation
# ------------------------------------------------------------

print("=" * 118)
print("STEP 1 - Q1: C/D POINT-BURDEN DECOMPOSITION IN S1-S3")
print("=" * 118)

print(
    "Sparse images                : 60"
)

print(
    "C sparse rows                :",
    len(
        c_sparse_q1_df
    ),
    "/ 180"
)

print(
    "D sparse rows                :",
    len(
        d_sparse_q1_df
    ),
    "/ 300"
)

print(
    "Combined diagnostic rows     :",
    len(
        q1_per_image_df
    ),
    "/ 480"
)

print(
    "TP + FP = pred_count         : PASS"
)

print(
    "TP + FN = gt_count           : PASS"
)

print(
    "Precision reconstruction     : PASS"
)

print(
    "Recall reconstruction        : PASS"
)

print(
    "New inference                : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nS1-S3 burden decomposition:"
)

display(
    display_by_stratum_df
)


print(
    "\nCombined S1-S3 summary:"
)

display(
    display_overall_df
)


print(
    "\nAggregate predicted-points / GT ratio:"
)

display(
    pred_gt_ratio_pivot
    .round(
        3
    )
)


print(
    "\nPooled precision:"
)

display(
    precision_pivot
    .round(
        3
    )
)


print(
    "\nPooled recall:"
)

display(
    recall_pivot
    .round(
        3
    )
)


print(
    "\nPooled FP per TP:"
)

display(
    fp_per_tp_pivot
    .round(
        3
    )
)


print(
    "\nSaved per-image diagnostic:",
    Q1_PER_IMAGE_FILE,
)

print(
    "Saved S1-S3 breakdown:",
    Q1_BY_STRATUM_FILE,
)

print(
    "Saved combined sparse summary:",
    Q1_OVERALL_SPARSE_FILE,
)

print("=" * 118)

STEP 1 - Q1: C/D POINT-BURDEN DECOMPOSITION IN S1-S3
Sparse images                : 60
C sparse rows                : 180 / 180
D sparse rows                : 300 / 300
Combined diagnostic rows     : 480 / 480
TP + FP = pred_count         : PASS
TP + FN = gt_count           : PASS
Precision reconstruction     : PASS
Recall reconstruction        : PASS
New inference                : NO
Threshold tuning             : NO
Test set accessed            : NO

S1-S3 burden decomposition:


,branch,candidate_id,density_stratum,mean_gt_count,mean_pred_count,aggregate_pred_gt_ratio,total_tp,total_fp,total_fn,pooled_precision,pooled_recall,pooled_f1,pooled_fp_per_tp,over_image_rate
0,C,C1,S1_0_20,9.75,12.00,1.2308,63,177,132,0.2625,0.3231,0.2897,2.8095,0.50
1,C,C1,S2_20_40,16.00,19.60,1.2250,94,298,226,0.2398,0.2938,0.2640,3.1702,0.45
2,C,C1,S3_40_60,26.85,20.05,0.7467,110,291,427,0.2743,0.2048,0.2345,2.6455,0.40
3,C,C2b,S1_0_20,9.75,37.95,3.8923,95,664,100,0.1252,0.4872,0.1992,6.9895,0.95
4,C,C2b,S2_20_40,16.00,58.50,3.6562,161,1009,159,0.1376,0.5031,0.2161,6.2671,1.00
5,C,C2b,S3_40_60,26.85,61.20,2.2793,246,978,291,0.2010,0.4581,0.2794,3.9756,0.90
6,C,C2c,S1_0_20,9.75,24.70,2.5333,84,410,111,0.1700,0.4308,0.2438,4.8810,0.90
7,C,C2c,S2_20_40,16.00,37.40,2.3375,150,598,170,0.2005,0.4688,0.2809,3.9867,0.95
8,C,C2c,S3_40_60,26.85,38.55,1.4358,219,552,318,0.2840,0.4078,0.3349,2.5205,0.75
9,D,D1,S1_0_20,9.75,54.10,5.5487,89,993,106,0.0823,0.4564,0.1394,11.1573,0.95



Combined S1-S3 summary:


,branch,candidate_id,mean_gt_count,mean_pred_count,aggregate_pred_gt_ratio,total_gt,total_pred,total_tp,total_fp,total_fn,pooled_precision,pooled_recall,pooled_f1,pooled_fp_per_tp,over_image_rate
0,C,C1,17.5333,17.2167,0.9819,1052,1033,267,766,785,0.2585,0.2538,0.2561,2.8689,0.4500
1,C,C2c,17.5333,33.5500,1.9135,1052,2013,453,1560,599,0.2250,0.4306,0.2956,3.4437,0.8667
2,C,C2b,17.5333,52.5500,2.9971,1052,3153,502,2651,550,0.1592,0.4772,0.2388,5.2809,0.9500
3,D,D4,17.5333,34.6167,1.9743,1052,2077,426,1651,626,0.2051,0.4049,0.2723,3.8756,0.8167
4,D,D6B9-strict,17.5333,39.9667,2.2795,1052,2398,487,1911,565,0.2031,0.4629,0.2823,3.9240,0.6333
5,D,D1,17.5333,61.7500,3.5219,1052,3705,513,3192,539,0.1385,0.4876,0.2157,6.2222,0.8333
6,D,D5-768,17.5333,116.4333,6.6407,1052,6986,616,6370,436,0.0882,0.5856,0.1533,10.3409,0.9833
7,D,D6B-T-native,17.5333,130.7667,7.4582,1052,7846,742,7104,310,0.0946,0.7053,0.1668,9.5741,0.9000



Aggregate predicted-points / GT ratio:


...


Pooled recall:


density_stratum      S1_0_20  S2_20_40  S3_40_60
branch candidate_id                             
C      C1              0.323     0.294     0.205
       C2b             0.487     0.503     0.458
       C2c             0.431     0.469     0.408
D      D1              0.456     0.534     0.471
       D4              0.395     0.444     0.385
       D5-768          0.518     0.653     0.570
       D6B-T-native    0.682     0.781     0.669
       D6B9-strict     0.492     0.569     0.389


Pooled FP per TP:


density_stratum      S1_0_20  S2_20_40  S3_40_60
branch candidate_id                             
C      C1              2.810     3.170     2.645
       C2b             6.989     6.267     3.976
       C2c             4.881     3.987     2.521
D      D1             11.157     6.211     4.494
       D4              7.104     3.915     2.647
       D5-768         18.802    11.177     6.977
       D6B-T-native   14.218    10.400     7.279
       D6B9-strict     4.510     3.681     3.866


Saved per-image diagnostic: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q1_CD_point_burden_per_image_S1_S3.csv
Saved S1-S3 breakdown: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q1_CD_point_burden_by_sparse_stratum.csv
Saved combined sparse summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q1_CD_point_burden_overall_S1_S3.csv


In [4]:
# ============================================================
# Step 2 - Q2: Point-Level Spatial Failure Taxonomy in S1-S3
# ============================================================

import ast
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2.1 Output files
# ------------------------------------------------------------

Q2_POINT_LEVEL_FILE = (
    DIAG_TABLES_DIR
    / "Q2_pred_point_spatial_taxonomy_S1_S3.csv"
)

Q2_IMAGE_LEVEL_FILE = (
    DIAG_TABLES_DIR
    / "Q2_spatial_taxonomy_per_image_S1_S3.csv"
)

Q2_BY_STRATUM_FILE = (
    DIAG_TABLES_DIR
    / "Q2_spatial_taxonomy_by_sparse_stratum.csv"
)

Q2_GT_MULTIPLICITY_FILE = (
    DIAG_TABLES_DIR
    / "Q2_GT_local_prediction_multiplicity_S1_S3.csv"
)

Q2_GT_MULTIPLICITY_SUMMARY_FILE = (
    DIAG_TABLES_DIR
    / "Q2_GT_local_prediction_multiplicity_summary.csv"
)


# ------------------------------------------------------------
# 2.2 Parse stored point lists
# ------------------------------------------------------------

def parse_xy_list(value):
    """
    Parse stored CSV point-list representation into
    float32 ndarray of shape (N, 2).

    Accepts:
      - string representation of Python list
      - list / tuple
      - ndarray
      - empty / NaN
    """

    if value is None:
        return np.empty(
            (0, 2),
            dtype=np.float32,
        )

    if isinstance(
        value,
        float,
    ) and np.isnan(
        value
    ):
        return np.empty(
            (0, 2),
            dtype=np.float32,
        )

    if isinstance(
        value,
        str,
    ):

        value = value.strip()

        if (
            value == ""
            or value == "[]"
        ):
            return np.empty(
                (0, 2),
                dtype=np.float32,
            )

        value = ast.literal_eval(
            value
        )


    arr = np.asarray(
        value,
        dtype=np.float32,
    )


    if arr.size == 0:
        return np.empty(
            (0, 2),
            dtype=np.float32,
        )


    arr = arr.reshape(
        -1,
        2,
    )

    return arr


# ------------------------------------------------------------
# 2.3 Prepare harmonised C + D sparse rows
# ------------------------------------------------------------

Q2_REQUIRED_COLUMNS = {
    "candidate_id",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "pred_count",
    "radius_px",
    "tp",
    "fp",
    "fn",
    "gt_points_xy",
    "pred_points_xy",
}


for df, name in [
    (c_sparse_q1_df, "C"),
    (d_sparse_q1_df, "D"),
]:

    missing = (
        Q2_REQUIRED_COLUMNS
        - set(
            df.columns
        )
    )

    assert (
        len(
            missing
        )
        == 0
    ), (
        f"{name} missing columns: "
        f"{sorted(missing)}"
    )


q2_source_df = (
    pd.concat(
        [
            c_sparse_q1_df,
            d_sparse_q1_df,
        ],
        ignore_index=True,
    )
)


assert (
    len(
        q2_source_df
    )
    == 480
)


# ------------------------------------------------------------
# 2.4 Point-level and GT-level spatial decomposition
# ------------------------------------------------------------

point_records = []
gt_records = []
image_records = []


for row in q2_source_df.itertuples(
    index=False
):

    gt_xy = parse_xy_list(
        row.gt_points_xy
    )

    pred_xy = parse_xy_list(
        row.pred_points_xy
    )

    radius_px = float(
        row.radius_px
    )


    # --------------------------------------------------------
    # Integrity
    # --------------------------------------------------------

    assert (
        len(
            gt_xy
        )
        == int(
            row.gt_count
        )
    ), (
        f"GT-point count mismatch: "
        f"{row.candidate_id}, "
        f"{row.image_id}"
    )


    assert (
        len(
            pred_xy
        )
        == int(
            row.pred_count
        )
    ), (
        f"Pred-point count mismatch: "
        f"{row.candidate_id}, "
        f"{row.image_id}"
    )


    assert (
        radius_px
        > 0
    )


    # --------------------------------------------------------
    # Pairwise Euclidean distance matrix
    #
    # Shape:
    #   (num_predictions, num_GT)
    # --------------------------------------------------------

    if (
        len(
            pred_xy
        )
        > 0
    ):

        distances = np.linalg.norm(
            pred_xy[
                :,
                None,
                :
            ]
            -
            gt_xy[
                None,
                :,
                :
            ],
            axis=2,
        )


        nearest_gt_index = np.argmin(
            distances,
            axis=1,
        )

        nearest_gt_distance_px = np.min(
            distances,
            axis=1,
        )

        nearest_gt_distance_R = (
            nearest_gt_distance_px
            /
            radius_px
        )


    else:

        distances = np.empty(
            (
                0,
                len(
                    gt_xy
                ),
            ),
            dtype=np.float32,
        )

        nearest_gt_index = np.empty(
            (0,),
            dtype=int,
        )

        nearest_gt_distance_px = np.empty(
            (0,),
            dtype=np.float32,
        )

        nearest_gt_distance_R = np.empty(
            (0,),
            dtype=np.float32,
        )


    # --------------------------------------------------------
    # Prediction-side spatial taxonomy
    #
    # <=1R:
    #   close enough to at least one GT according to the
    #   frozen localisation tolerance.
    #
    # >1R and <=2R:
    #   near-object evidence, but outside matching radius.
    #
    # >2R:
    #   spatially remote from all GT.
    # --------------------------------------------------------

    spatial_class = np.empty(
        len(
            pred_xy
        ),
        dtype=object,
    )


    for pred_idx in range(
        len(
            pred_xy
        )
    ):

        norm_dist = float(
            nearest_gt_distance_R[
                pred_idx
            ]
        )


        if norm_dist <= 1.0:

            cls = "within_1R"

        elif norm_dist <= 2.0:

            cls = "between_1R_2R"

        else:

            cls = "beyond_2R"


        spatial_class[
            pred_idx
        ] = cls


        point_records.append(
            {
                "branch":
                    row.branch,

                "candidate_id":
                    row.candidate_id,

                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    row.image_id,

                "category":
                    row.category,

                "density_stratum":
                    row.density_stratum,

                "gt_count":
                    int(
                        row.gt_count
                    ),

                "pred_count":
                    int(
                        row.pred_count
                    ),

                "radius_px":
                    radius_px,

                "pred_index":
                    int(
                        pred_idx
                    ),

                "pred_x":
                    float(
                        pred_xy[
                            pred_idx,
                            0
                        ]
                    ),

                "pred_y":
                    float(
                        pred_xy[
                            pred_idx,
                            1
                        ]
                    ),

                "nearest_gt_index":
                    int(
                        nearest_gt_index[
                            pred_idx
                        ]
                    ),

                "nearest_gt_distance_px":
                    float(
                        nearest_gt_distance_px[
                            pred_idx
                        ]
                    ),

                "nearest_gt_distance_R":
                    norm_dist,

                "spatial_class":
                    cls,
            }
        )


    # --------------------------------------------------------
    # Prediction-side counts for this image/candidate
    # --------------------------------------------------------

    n_within_1R = int(
        np.sum(
            nearest_gt_distance_R
            <= 1.0
        )
    )

    n_between_1R_2R = int(
        np.sum(
            (
                nearest_gt_distance_R
                > 1.0
            )
            &
            (
                nearest_gt_distance_R
                <= 2.0
            )
        )
    )

    n_beyond_2R = int(
        np.sum(
            nearest_gt_distance_R
            > 2.0
        )
    )


    assert (
        n_within_1R
        +
        n_between_1R_2R
        +
        n_beyond_2R
        ==
        int(
            row.pred_count
        )
    )


    # --------------------------------------------------------
    # A TP must have been within the frozen match radius.
    #
    # Therefore:
    #
    # within_1R - TP
    #
    # gives the number of geometrically-close predictions
    # which could not be retained as unique one-to-one TPs.
    #
    # We call this "local surplus". It is diagnostic, not a
    # standard metric and not necessarily literal duplicates.
    # --------------------------------------------------------

    local_surplus_within_1R = (
        n_within_1R
        -
        int(
            row.tp
        )
    )


    assert (
        local_surplus_within_1R
        >= 0
    ), (
        f"TP exceeds number of predictions within radius: "
        f"{row.candidate_id}, "
        f"{row.image_id}"
    )


    # --------------------------------------------------------
    # GT-side local multiplicity
    #
    # Count how many predicted points fall inside radius R
    # around each GT point.
    #
    # Note:
    # A prediction may fall inside R of more than one GT
    # when GT points are close together. Therefore this is
    # a local-support diagnostic, not one-to-one matching.
    # --------------------------------------------------------

    if len(
        pred_xy
    ) > 0:

        pred_inside_each_gt = (
            distances
            <= radius_px
        ).sum(
            axis=0
        )

        nearest_pred_to_gt_px = (
            distances.min(
                axis=0
            )
        )

        nearest_pred_to_gt_R = (
            nearest_pred_to_gt_px
            /
            radius_px
        )


    else:

        pred_inside_each_gt = np.zeros(
            len(
                gt_xy
            ),
            dtype=int,
        )

        nearest_pred_to_gt_px = np.full(
            len(
                gt_xy
            ),
            np.inf,
        )

        nearest_pred_to_gt_R = np.full(
            len(
                gt_xy
            ),
            np.inf,
        )


    for gt_idx in range(
        len(
            gt_xy
        )
    ):

        multiplicity = int(
            pred_inside_each_gt[
                gt_idx
            ]
        )


        if multiplicity == 0:

            multiplicity_bin = "0"

        elif multiplicity == 1:

            multiplicity_bin = "1"

        elif multiplicity == 2:

            multiplicity_bin = "2"

        elif multiplicity == 3:

            multiplicity_bin = "3"

        elif multiplicity == 4:

            multiplicity_bin = "4"

        else:

            multiplicity_bin = "5+"


        gt_records.append(
            {
                "branch":
                    row.branch,

                "candidate_id":
                    row.candidate_id,

                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    row.image_id,

                "category":
                    row.category,

                "density_stratum":
                    row.density_stratum,

                "radius_px":
                    radius_px,

                "gt_index":
                    int(
                        gt_idx
                    ),

                "gt_x":
                    float(
                        gt_xy[
                            gt_idx,
                            0
                        ]
                    ),

                "gt_y":
                    float(
                        gt_xy[
                            gt_idx,
                            1
                        ]
                    ),

                "num_predictions_within_1R":
                    multiplicity,

                "multiplicity_bin":
                    multiplicity_bin,

                "nearest_prediction_distance_px":
                    float(
                        nearest_pred_to_gt_px[
                            gt_idx
                        ]
                    ),

                "nearest_prediction_distance_R":
                    float(
                        nearest_pred_to_gt_R[
                            gt_idx
                        ]
                    ),
            }
        )


    # --------------------------------------------------------
    # GT multiplicity summary for image
    # --------------------------------------------------------

    gt_zero_local = int(
        np.sum(
            pred_inside_each_gt
            == 0
        )
    )

    gt_one_local = int(
        np.sum(
            pred_inside_each_gt
            == 1
        )
    )

    gt_multi_local = int(
        np.sum(
            pred_inside_each_gt
            >= 2
        )
    )


    # --------------------------------------------------------
    # Image-level record
    # --------------------------------------------------------

    image_records.append(
        {
            "branch":
                row.branch,

            "candidate_id":
                row.candidate_id,

            "dataset_index":
                int(
                    row.dataset_index
                ),

            "image_id":
                row.image_id,

            "category":
                row.category,

            "density_stratum":
                row.density_stratum,

            "gt_count":
                int(
                    row.gt_count
                ),

            "pred_count":
                int(
                    row.pred_count
                ),

            "tp":
                int(
                    row.tp
                ),

            "fp":
                int(
                    row.fp
                ),

            "fn":
                int(
                    row.fn
                ),

            "radius_px":
                radius_px,

            # ----------------------------------------------
            # Prediction-side spatial taxonomy
            # ----------------------------------------------

            "pred_within_1R":
                n_within_1R,

            "pred_between_1R_2R":
                n_between_1R_2R,

            "pred_beyond_2R":
                n_beyond_2R,

            "local_surplus_within_1R":
                local_surplus_within_1R,

            "rate_pred_within_1R":
                (
                    n_within_1R
                    /
                    row.pred_count
                    if row.pred_count > 0
                    else 0.0
                ),

            "rate_pred_between_1R_2R":
                (
                    n_between_1R_2R
                    /
                    row.pred_count
                    if row.pred_count > 0
                    else 0.0
                ),

            "rate_pred_beyond_2R":
                (
                    n_beyond_2R
                    /
                    row.pred_count
                    if row.pred_count > 0
                    else 0.0
                ),

            "rate_local_surplus_within_1R":
                (
                    local_surplus_within_1R
                    /
                    row.pred_count
                    if row.pred_count > 0
                    else 0.0
                ),

            # ----------------------------------------------
            # GT-side local multiplicity
            # ----------------------------------------------

            "gt_zero_predictions_within_1R":
                gt_zero_local,

            "gt_one_prediction_within_1R":
                gt_one_local,

            "gt_multiple_predictions_within_1R":
                gt_multi_local,

            "gt_zero_local_rate":
                (
                    gt_zero_local
                    /
                    row.gt_count
                ),

            "gt_one_local_rate":
                (
                    gt_one_local
                    /
                    row.gt_count
                ),

            "gt_multiple_local_rate":
                (
                    gt_multi_local
                    /
                    row.gt_count
                ),
        }
    )


# ------------------------------------------------------------
# 2.5 Build dataframes
# ------------------------------------------------------------

q2_point_df = pd.DataFrame(
    point_records
)

q2_gt_df = pd.DataFrame(
    gt_records
)

q2_image_df = pd.DataFrame(
    image_records
)


assert (
    len(
        q2_image_df
    )
    == 480
)


# Total number of point-level rows must equal
# total number of predictions in C+D sparse data.

assert (
    len(
        q2_point_df
    )
    ==
    int(
        q2_source_df[
            "pred_count"
        ].sum()
    )
)


# GT rows = each GT repeated once for each of 8 candidates.

assert (
    len(
        q2_gt_df
    )
    ==
    (
        int(
            sparse_dev_df[
                "gt_count"
            ].sum()
        )
        *
        8
    )
)


# ------------------------------------------------------------
# 2.6 Reaggregation integrity checks
# ------------------------------------------------------------

assert (
    (
        q2_image_df[
            "pred_within_1R"
        ]
        +
        q2_image_df[
            "pred_between_1R_2R"
        ]
        +
        q2_image_df[
            "pred_beyond_2R"
        ]
    )
    ==
    q2_image_df[
        "pred_count"
    ]
).all()


assert (
    (
        q2_image_df[
            "gt_zero_predictions_within_1R"
        ]
        +
        q2_image_df[
            "gt_one_prediction_within_1R"
        ]
        +
        q2_image_df[
            "gt_multiple_predictions_within_1R"
        ]
    )
    ==
    q2_image_df[
        "gt_count"
    ]
).all()


# ------------------------------------------------------------
# 2.7 Aggregate prediction-side taxonomy by stratum
# ------------------------------------------------------------

taxonomy_records = []


for (
    branch,
    candidate_id,
    density_stratum,
), group in q2_image_df.groupby(
    [
        "branch",
        "candidate_id",
        "density_stratum",
    ],
    sort=False,
):

    total_pred = int(
        group[
            "pred_count"
        ].sum()
    )

    total_tp = int(
        group[
            "tp"
        ].sum()
    )

    total_fp = int(
        group[
            "fp"
        ].sum()
    )

    total_within_1R = int(
        group[
            "pred_within_1R"
        ].sum()
    )

    total_between = int(
        group[
            "pred_between_1R_2R"
        ].sum()
    )

    total_remote = int(
        group[
            "pred_beyond_2R"
        ].sum()
    )

    total_local_surplus = int(
        group[
            "local_surplus_within_1R"
        ].sum()
    )


    total_gt = int(
        group[
            "gt_count"
        ].sum()
    )

    gt_zero = int(
        group[
            "gt_zero_predictions_within_1R"
        ].sum()
    )

    gt_one = int(
        group[
            "gt_one_prediction_within_1R"
        ].sum()
    )

    gt_multi = int(
        group[
            "gt_multiple_predictions_within_1R"
        ].sum()
    )


    taxonomy_records.append(
        {
            "branch":
                branch,

            "candidate_id":
                candidate_id,

            "density_stratum":
                density_stratum,

            "num_images":
                int(
                    len(
                        group
                    )
                ),

            "total_gt":
                total_gt,

            "total_pred":
                total_pred,

            "total_tp":
                total_tp,

            "total_fp":
                total_fp,

            # ----------------------------------------------
            # Prediction-side geometry
            # ----------------------------------------------

            "pred_within_1R":
                total_within_1R,

            "pred_between_1R_2R":
                total_between,

            "pred_beyond_2R":
                total_remote,

            "local_surplus_within_1R":
                total_local_surplus,

            "rate_pred_within_1R":
                (
                    total_within_1R
                    /
                    total_pred
                    if total_pred > 0
                    else 0.0
                ),

            "rate_pred_between_1R_2R":
                (
                    total_between
                    /
                    total_pred
                    if total_pred > 0
                    else 0.0
                ),

            "rate_pred_beyond_2R":
                (
                    total_remote
                    /
                    total_pred
                    if total_pred > 0
                    else 0.0
                ),

            # What proportion of all frozen one-to-one FPs
            # are nevertheless geometrically within 1R?
            "local_surplus_as_fraction_of_FP":
                (
                    total_local_surplus
                    /
                    total_fp
                    if total_fp > 0
                    else 0.0
                ),

            # ----------------------------------------------
            # GT-side local multiplicity
            # ----------------------------------------------

            "gt_zero_local":
                gt_zero,

            "gt_one_local":
                gt_one,

            "gt_multiple_local":
                gt_multi,

            "gt_zero_local_rate":
                (
                    gt_zero
                    /
                    total_gt
                ),

            "gt_one_local_rate":
                (
                    gt_one
                    /
                    total_gt
                ),

            "gt_multiple_local_rate":
                (
                    gt_multi
                    /
                    total_gt
                ),
        }
    )


q2_by_stratum_df = pd.DataFrame(
    taxonomy_records
)


q2_by_stratum_df[
    "density_stratum"
] = pd.Categorical(
    q2_by_stratum_df[
        "density_stratum"
    ],
    categories=SPARSE_STRATA,
    ordered=True,
)


q2_by_stratum_df = (
    q2_by_stratum_df
    .sort_values(
        [
            "branch",
            "candidate_id",
            "density_stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        q2_by_stratum_df
    )
    == 24
)


# ------------------------------------------------------------
# 2.8 GT multiplicity-bin summary
# ------------------------------------------------------------

MULTIPLICITY_ORDER = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5+",
]


gt_mult_summary_records = []


for (
    branch,
    candidate_id,
    density_stratum,
), group in q2_gt_df.groupby(
    [
        "branch",
        "candidate_id",
        "density_stratum",
    ],
    sort=False,
):

    total_gt = int(
        len(
            group
        )
    )


    counts = (
        group[
            "multiplicity_bin"
        ]
        .value_counts()
        .reindex(
            MULTIPLICITY_ORDER,
            fill_value=0,
        )
    )


    record = {
        "branch":
            branch,

        "candidate_id":
            candidate_id,

        "density_stratum":
            density_stratum,

        "total_gt":
            total_gt,

        "mean_predictions_within_1R_per_GT":
            float(
                group[
                    "num_predictions_within_1R"
                ].mean()
            ),

        "median_predictions_within_1R_per_GT":
            float(
                group[
                    "num_predictions_within_1R"
                ].median()
            ),

        "max_predictions_within_1R_per_GT":
            int(
                group[
                    "num_predictions_within_1R"
                ].max()
            ),
    }


    for bin_name in MULTIPLICITY_ORDER:

        n = int(
            counts[
                bin_name
            ]
        )

        record[
            f"gt_mult_{bin_name}_count"
        ] = n

        record[
            f"gt_mult_{bin_name}_rate"
        ] = (
            n
            /
            total_gt
        )


    gt_mult_summary_records.append(
        record
    )


q2_gt_multiplicity_summary_df = pd.DataFrame(
    gt_mult_summary_records
)


q2_gt_multiplicity_summary_df[
    "density_stratum"
] = pd.Categorical(
    q2_gt_multiplicity_summary_df[
        "density_stratum"
    ],
    categories=SPARSE_STRATA,
    ordered=True,
)


q2_gt_multiplicity_summary_df = (
    q2_gt_multiplicity_summary_df
    .sort_values(
        [
            "branch",
            "candidate_id",
            "density_stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 2.9 Save outputs
# ------------------------------------------------------------

q2_point_df.to_csv(
    Q2_POINT_LEVEL_FILE,
    index=False,
)

q2_image_df.to_csv(
    Q2_IMAGE_LEVEL_FILE,
    index=False,
)

q2_by_stratum_df.to_csv(
    Q2_BY_STRATUM_FILE,
    index=False,
)

q2_gt_df.to_csv(
    Q2_GT_MULTIPLICITY_FILE,
    index=False,
)

q2_gt_multiplicity_summary_df.to_csv(
    Q2_GT_MULTIPLICITY_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 2.10 Compact display tables
# ------------------------------------------------------------

TAXONOMY_DISPLAY_COLS = [
    "branch",
    "candidate_id",
    "density_stratum",
    "total_gt",
    "total_pred",
    "total_tp",
    "total_fp",
    "rate_pred_within_1R",
    "rate_pred_between_1R_2R",
    "rate_pred_beyond_2R",
    "local_surplus_as_fraction_of_FP",
    "gt_zero_local_rate",
    "gt_one_local_rate",
    "gt_multiple_local_rate",
]


taxonomy_display_df = (
    q2_by_stratum_df[
        TAXONOMY_DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "rate_pred_within_1R",
    "rate_pred_between_1R_2R",
    "rate_pred_beyond_2R",
    "local_surplus_as_fraction_of_FP",
    "gt_zero_local_rate",
    "gt_one_local_rate",
    "gt_multiple_local_rate",
]:

    taxonomy_display_df[
        col
    ] = (
        taxonomy_display_df[
            col
        ]
        .round(
            4
        )
    )


MULT_DISPLAY_COLS = [
    "branch",
    "candidate_id",
    "density_stratum",
    "total_gt",
    "mean_predictions_within_1R_per_GT",
    "median_predictions_within_1R_per_GT",
    "max_predictions_within_1R_per_GT",
    "gt_mult_0_rate",
    "gt_mult_1_rate",
    "gt_mult_2_rate",
    "gt_mult_3_rate",
    "gt_mult_4_rate",
    "gt_mult_5+_rate",
]


multiplicity_display_df = (
    q2_gt_multiplicity_summary_df[
        MULT_DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "mean_predictions_within_1R_per_GT",
    "median_predictions_within_1R_per_GT",
    "gt_mult_0_rate",
    "gt_mult_1_rate",
    "gt_mult_2_rate",
    "gt_mult_3_rate",
    "gt_mult_4_rate",
    "gt_mult_5+_rate",
]:

    multiplicity_display_df[
        col
    ] = (
        multiplicity_display_df[
            col
        ]
        .round(
            4
        )
    )


# ------------------------------------------------------------
# 2.11 Pivots for the two central diagnostic questions
# ------------------------------------------------------------

remote_rate_pivot = (
    q2_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="rate_pred_beyond_2R",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


local_surplus_fp_pivot = (
    q2_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="local_surplus_as_fraction_of_FP",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


gt_multi_pivot = (
    q2_by_stratum_df
    .pivot(
        index=[
            "branch",
            "candidate_id",
        ],
        columns="density_stratum",
        values="gt_multiple_local_rate",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


# ------------------------------------------------------------
# 2.12 Confirmation
# ------------------------------------------------------------

print("=" * 120)
print("STEP 2 - Q2: POINT-LEVEL SPATIAL FAILURE TAXONOMY IN S1-S3")
print("=" * 120)

print(
    "Image/candidate rows         :",
    len(
        q2_image_df
    ),
    "/ 480"
)

print(
    "Prediction-level rows        :",
    len(
        q2_point_df
    )
)

print(
    "Expected prediction rows     :",
    int(
        q2_source_df[
            "pred_count"
        ].sum()
    )
)

print(
    "GT-level rows                :",
    len(
        q2_gt_df
    )
)

print(
    "Expected GT rows             :",
    int(
        sparse_dev_df[
            "gt_count"
        ].sum()
    )
    * 8
)

print(
    "Point-list count integrity   : PASS"
)

print(
    "Spatial-bin reaggregation    : PASS"
)

print(
    "GT multiplicity reaggregation: PASS"
)

print(
    "New inference                : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nPrediction-side + GT-side spatial taxonomy:"
)

display(
    taxonomy_display_df
)


print(
    "\nGT local prediction multiplicity:"
)

display(
    multiplicity_display_df
)


print(
    "\nRate of predicted points >2R from every GT:"
)

display(
    remote_rate_pivot
    .round(
        3
    )
)


print(
    "\nFraction of one-to-one FPs that are nevertheless within 1R of some GT:"
)

display(
    local_surplus_fp_pivot
    .round(
        3
    )
)


print(
    "\nGT rate with >=2 predicted points inside 1R:"
)

display(
    gt_multi_pivot
    .round(
        3
    )
)


print(
    "\nSaved prediction-level taxonomy:",
    Q2_POINT_LEVEL_FILE,
)

print(
    "Saved image-level taxonomy:",
    Q2_IMAGE_LEVEL_FILE,
)

print(
    "Saved stratum summary:",
    Q2_BY_STRATUM_FILE,
)

print(
    "Saved GT multiplicity records:",
    Q2_GT_MULTIPLICITY_FILE,
)

print(
    "Saved GT multiplicity summary:",
    Q2_GT_MULTIPLICITY_SUMMARY_FILE,
)

print("=" * 120)

STEP 2 - Q2: POINT-LEVEL SPATIAL FAILURE TAXONOMY IN S1-S3
Image/candidate rows         : 480 / 480
Prediction-level rows        : 29211
Expected prediction rows     : 29211
GT-level rows                : 8416
Expected GT rows             : 8416
Point-list count integrity   : PASS
Spatial-bin reaggregation    : PASS
GT multiplicity reaggregation: PASS
New inference                : NO
Threshold tuning             : NO
Test set accessed            : NO

Prediction-side + GT-side spatial taxonomy:


,branch,candidate_id,density_stratum,total_gt,total_pred,total_tp,total_fp,rate_pred_within_1R,rate_pred_between_1R_2R,rate_pred_beyond_2R,local_surplus_as_fraction_of_FP,gt_zero_local_rate,gt_one_local_rate,gt_multiple_local_rate
0,C,C1,S1_0_20,195,240,63,177,0.2625,0.1833,0.5542,0.0000,0.6769,0.3231,0.0000
1,C,C1,S2_20_40,320,392,94,298,0.2449,0.1760,0.5791,0.0067,0.6969,0.2969,0.0062
2,C,C1,S3_40_60,537,401,110,291,0.2743,0.1820,0.5436,0.0000,0.7877,0.2123,0.0000
3,C,C2b,S1_0_20,195,759,95,664,0.1542,0.1989,0.6469,0.0331,0.5128,0.3846,0.1026
4,C,C2b,S2_20_40,320,1170,161,1009,0.1624,0.1513,0.6863,0.0287,0.4812,0.4375,0.0812
5,C,C2b,S3_40_60,537,1224,246,978,0.2500,0.2108,0.5392,0.0613,0.5121,0.3706,0.1173
6,C,C2c,S1_0_20,195,494,84,410,0.1781,0.1943,0.6275,0.0098,0.5692,0.3949,0.0359
7,C,C2c,S2_20_40,320,748,150,598,0.2086,0.1444,0.6471,0.0100,0.5125,0.4688,0.0188
8,C,C2c,S3_40_60,537,771,219,552,0.2931,0.1842,0.5227,0.0127,0.5493,0.4283,0.0223
9,D,D1,S1_0_20,195,1082,89,993,0.0850,0.1784,0.7366,0.0030,0.5385,0.4462,0.0154



GT local prediction multiplicity:


,branch,candidate_id,density_stratum,total_gt,mean_predictions_within_1R_per_GT,median_predictions_within_1R_per_GT,max_predictions_within_1R_per_GT,gt_mult_0_rate,gt_mult_1_rate,gt_mult_2_rate,gt_mult_3_rate,gt_mult_4_rate,gt_mult_5+_rate
0,C,C1,S1_0_20,195,0.3231,0.0,1,0.6769,0.3231,0.0000,0.0000,0.0000,0.0
1,C,C1,S2_20_40,320,0.3094,0.0,2,0.6969,0.2969,0.0062,0.0000,0.0000,0.0
2,C,C1,S3_40_60,537,0.2123,0.0,1,0.7877,0.2123,0.0000,0.0000,0.0000,0.0
3,C,C2b,S1_0_20,195,0.6205,0.0,4,0.5128,0.3846,0.0769,0.0205,0.0051,0.0
4,C,C2b,S2_20_40,320,0.6094,1.0,3,0.4812,0.4375,0.0719,0.0094,0.0000,0.0
5,C,C2b,S3_40_60,537,0.6462,0.0,4,0.5121,0.3706,0.0838,0.0261,0.0074,0.0
6,C,C2c,S1_0_20,195,0.4667,0.0,2,0.5692,0.3949,0.0359,0.0000,0.0000,0.0
7,C,C2c,S2_20_40,320,0.5062,0.0,2,0.5125,0.4688,0.0188,0.0000,0.0000,0.0
8,C,C2c,S3_40_60,537,0.4730,0.0,2,0.5493,0.4283,0.0223,0.0000,0.0000,0.0
9,D,D1,S1_0_20,195,0.4769,0.0,2,0.5385,0.4462,0.0154,0.0000,0.0000,0.0



Rate of predicted points >2R from every GT:


...


Fraction of one-to-one FPs that are nevertheless within 1R of some GT:


density_stratum      S1_0_20  S2_20_40  S3_40_60
branch candidate_id                             
C      C1              0.000     0.007     0.000
       C2b             0.033     0.029     0.061
       C2c             0.010     0.010     0.013
D      D1              0.003     0.012     0.006
       D4              0.005     0.020     0.007
       D5-768          0.004     0.010     0.015
       D6B-T-native    0.009     0.017     0.023
       D6B9-strict     0.009     0.015     0.017


GT rate with >=2 predicted points inside 1R:


density_stratum      S1_0_20  S2_20_40  S3_40_60
branch candidate_id                             
C      C1              0.000     0.006     0.000
       C2b             0.103     0.081     0.117
       C2c             0.036     0.019     0.022
D      D1              0.015     0.044     0.022
       D4              0.015     0.038     0.017
       D5-768          0.041     0.078     0.076
       D6B-T-native    0.087     0.134     0.123
       D6B9-strict     0.021     0.034     0.026


Saved prediction-level taxonomy: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q2_pred_point_spatial_taxonomy_S1_S3.csv
Saved image-level taxonomy: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q2_spatial_taxonomy_per_image_S1_S3.csv
Saved stratum summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q2_spatial_taxonomy_by_sparse_stratum.csv
Saved GT multiplicity records: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q2_GT_local_prediction_multiplicity_S1_S3.csv
Saved GT multiplicity summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q2_GT_local_prediction_multiplicity_summary.csv


In [5]:
# ============================================================
# Step 3 - Q3: Fusion Marginal-Value Analysis
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 3.1 Output files
# ------------------------------------------------------------

Q3_E2_COMPLEMENTARITY_FILE = (
    DIAG_TABLES_DIR
    / "Q3_E2_complementarity_sparse.csv"
)

Q3_FROZEN_FUSION_PER_IMAGE_FILE = (
    DIAG_TABLES_DIR
    / "Q3_frozen_fusion_marginal_value_per_image.csv"
)

Q3_FROZEN_FUSION_SUMMARY_FILE = (
    DIAG_TABLES_DIR
    / "Q3_frozen_fusion_marginal_value_summary.csv"
)


# ------------------------------------------------------------
# 3.2 Frozen Group-E representatives
# ------------------------------------------------------------

FROZEN_FUSIONS = {
    "overall_joint": {
        "fusion_family": "F2",
        "c_variant": "C2b",
        "d_variant": "D6B-T-native",
        "tau": 0.010,
    },

    "sparse_joint": {
        "fusion_family": "F2",
        "c_variant": "C2b",
        "d_variant": "D6B-T-native",
        "tau": 0.005,
    },

    "dense_recovery": {
        "fusion_family": "F1",
        "c_variant": "C1",
        "d_variant": "D6B9-strict",
        "tau": 0.050,
    },
}


# ------------------------------------------------------------
# 3.3 Required-column checks
# ------------------------------------------------------------

E2_REQUIRED = {
    "dataset_index",
    "image_id",
    "density_stratum",
    "c_variant",
    "d_variant",
    "both",
    "c_only",
    "d_only",
    "neither",
    "oracle_union_tp",
}

E3_REQUIRED = {
    "dataset_index",
    "image_id",
    "density_stratum",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
    "pred_count",
    "tp",
    "fp",
    "fn",
}


missing_e2 = (
    E2_REQUIRED
    - set(
        e2_df.columns
    )
)

missing_e3 = (
    E3_REQUIRED
    - set(
        e3_eval_df.columns
    )
)


assert (
    len(
        missing_e2
    )
    == 0
), (
    f"E2 missing required columns: "
    f"{sorted(missing_e2)}"
)


assert (
    len(
        missing_e3
    )
    == 0
), (
    f"E3 missing required columns: "
    f"{sorted(missing_e3)}"
)


# ------------------------------------------------------------
# 3.4 Sparse-only E2 complementarity
# ------------------------------------------------------------

e2_sparse_df = (
    e2_df[
        e2_df[
            "density_stratum"
        ].isin(
            SPARSE_STRATA
        )
    ]
    .copy()
)


assert (
    len(
        e2_sparse_df
    )
    == 15 * 60
)


# ------------------------------------------------------------
# 3.5 E2 integrity
#
# both + c_only + d_only + neither should equal GT count.
# If gt_count is present, check it directly.
# ------------------------------------------------------------

if "gt_count" in e2_sparse_df.columns:

    assert (
        (
            e2_sparse_df[
                "both"
            ]
            +
            e2_sparse_df[
                "c_only"
            ]
            +
            e2_sparse_df[
                "d_only"
            ]
            +
            e2_sparse_df[
                "neither"
            ]
        )
        ==
        e2_sparse_df[
            "gt_count"
        ]
    ).all()


# oracle union TP should equal all GT covered by either C or D

assert (
    e2_sparse_df[
        "oracle_union_tp"
    ]
    ==
    (
        e2_sparse_df[
            "both"
        ]
        +
        e2_sparse_df[
            "c_only"
        ]
        +
        e2_sparse_df[
            "d_only"
        ]
    )
).all()


# ------------------------------------------------------------
# 3.6 Summarise E2 complementarity
# ------------------------------------------------------------

e2_summary_records = []


for (
    c_variant,
    d_variant,
    density_stratum,
), group in e2_sparse_df.groupby(
    [
        "c_variant",
        "d_variant",
        "density_stratum",
    ],
    sort=False,
):

    both_sum = int(
        group[
            "both"
        ].sum()
    )

    c_only_sum = int(
        group[
            "c_only"
        ].sum()
    )

    d_only_sum = int(
        group[
            "d_only"
        ].sum()
    )

    neither_sum = int(
        group[
            "neither"
        ].sum()
    )

    oracle_union_tp_sum = int(
        group[
            "oracle_union_tp"
        ].sum()
    )

    gt_total = (
        both_sum
        + c_only_sum
        + d_only_sum
        + neither_sum
    )


    # C alone covers:
    # both + c_only
    c_covered = (
        both_sum
        + c_only_sum
    )

    # D alone covers:
    # both + d_only
    d_covered = (
        both_sum
        + d_only_sum
    )


    e2_summary_records.append(
        {
            "c_variant":
                c_variant,

            "d_variant":
                d_variant,

            "density_stratum":
                density_stratum,

            "num_images":
                int(
                    len(
                        group
                    )
                ),

            "total_gt":
                gt_total,

            "both":
                both_sum,

            "c_only":
                c_only_sum,

            "d_only":
                d_only_sum,

            "neither":
                neither_sum,

            "c_covered":
                c_covered,

            "d_covered":
                d_covered,

            "oracle_union_tp":
                oracle_union_tp_sum,

            "c_recall_from_e2":
                (
                    c_covered
                    /
                    gt_total
                ),

            "d_recall_from_e2":
                (
                    d_covered
                    /
                    gt_total
                ),

            "oracle_union_recall":
                (
                    oracle_union_tp_sum
                    /
                    gt_total
                ),

            # ----------------------------------------------
            # Marginal complementarity
            # ----------------------------------------------

            "d_only_rate_of_gt":
                (
                    d_only_sum
                    /
                    gt_total
                ),

            "c_only_rate_of_gt":
                (
                    c_only_sum
                    /
                    gt_total
                ),

            # Among all GT that D covers,
            # how much is genuinely unavailable from C?
            "d_unique_fraction_of_d_coverage":
                (
                    d_only_sum
                    /
                    d_covered
                    if d_covered > 0
                    else 0.0
                ),

            # Among all GT that C covers,
            # how much is unavailable from D?
            "c_unique_fraction_of_c_coverage":
                (
                    c_only_sum
                    /
                    c_covered
                    if c_covered > 0
                    else 0.0
                ),
        }
    )


q3_e2_summary_df = pd.DataFrame(
    e2_summary_records
)


q3_e2_summary_df[
    "density_stratum"
] = pd.Categorical(
    q3_e2_summary_df[
        "density_stratum"
    ],
    categories=SPARSE_STRATA,
    ordered=True,
)


q3_e2_summary_df = (
    q3_e2_summary_df
    .sort_values(
        [
            "c_variant",
            "d_variant",
            "density_stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 3.7 Build exact parent tables for C and D
# ------------------------------------------------------------

c_parent_cols = [
    "dataset_index",
    "image_id",
    "density_stratum",
    "candidate_id",
    "gt_count",
    "pred_count",
    "tp",
    "fp",
    "fn",
]

d_parent_cols = c_parent_cols.copy()


c_parent_df = (
    c_df[
        c_parent_cols
    ]
    .rename(
        columns={
            "candidate_id":
                "c_variant",

            "pred_count":
                "c_pred_count",

            "tp":
                "c_tp",

            "fp":
                "c_fp",

            "fn":
                "c_fn",
        }
    )
)


d_parent_df = (
    d_df[
        d_parent_cols
    ]
    .rename(
        columns={
            "candidate_id":
                "d_variant",

            "pred_count":
                "d_pred_count",

            "tp":
                "d_tp",

            "fp":
                "d_fp",

            "fn":
                "d_fn",
        }
    )
)


# ------------------------------------------------------------
# 3.8 Extract frozen E3 representatives
# ------------------------------------------------------------

frozen_fusion_rows = []


for checkpoint_id, cfg in FROZEN_FUSIONS.items():

    selected = (
        e3_eval_df[
            (
                e3_eval_df[
                    "fusion_family"
                ]
                == cfg[
                    "fusion_family"
                ]
            )
            &
            (
                e3_eval_df[
                    "c_variant"
                ]
                == cfg[
                    "c_variant"
                ]
            )
            &
            (
                e3_eval_df[
                    "d_variant"
                ]
                == cfg[
                    "d_variant"
                ]
            )
            &
            (
                np.isclose(
                    e3_eval_df[
                        "tau"
                    ],
                    cfg[
                        "tau"
                    ],
                )
            )
        ]
        .copy()
    )


    assert (
        len(
            selected
        )
        == 100
    ), (
        f"{checkpoint_id}: expected 100 E3 rows, "
        f"found {len(selected)}"
    )


    selected[
        "checkpoint_id"
    ] = checkpoint_id


    frozen_fusion_rows.append(
        selected
    )


frozen_e3_df = (
    pd.concat(
        frozen_fusion_rows,
        ignore_index=True,
    )
)


assert (
    len(
        frozen_e3_df
    )
    == 300
)


# ------------------------------------------------------------
# 3.9 Attach exact C and D parents
# ------------------------------------------------------------

q3_fusion_df = (
    frozen_e3_df
    .merge(
        c_parent_df,
        on=[
            "dataset_index",
            "image_id",
            "density_stratum",
            "c_variant",
            "gt_count",
        ],
        how="left",
        validate="many_to_one",
    )
    .merge(
        d_parent_df,
        on=[
            "dataset_index",
            "image_id",
            "density_stratum",
            "d_variant",
            "gt_count",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    q3_fusion_df[
        [
            "c_pred_count",
            "c_tp",
            "c_fp",
            "d_pred_count",
            "d_tp",
            "d_fp",
        ]
    ]
    .notna()
    .all()
    .all()
)


# ------------------------------------------------------------
# 3.10 Rename fused columns explicitly
# ------------------------------------------------------------

q3_fusion_df = (
    q3_fusion_df
    .rename(
        columns={
            "pred_count":
                "e_pred_count",

            "tp":
                "e_tp",

            "fp":
                "e_fp",

            "fn":
                "e_fn",
        }
    )
)


# ------------------------------------------------------------
# 3.11 Marginal effect of fusion relative to each parent
# ------------------------------------------------------------

for parent in [
    "c",
    "d",
]:

    q3_fusion_df[
        f"delta_pred_vs_{parent}"
    ] = (
        q3_fusion_df[
            "e_pred_count"
        ]
        -
        q3_fusion_df[
            f"{parent}_pred_count"
        ]
    )

    q3_fusion_df[
        f"delta_tp_vs_{parent}"
    ] = (
        q3_fusion_df[
            "e_tp"
        ]
        -
        q3_fusion_df[
            f"{parent}_tp"
        ]
    )

    q3_fusion_df[
        f"delta_fp_vs_{parent}"
    ] = (
        q3_fusion_df[
            "e_fp"
        ]
        -
        q3_fusion_df[
            f"{parent}_fp"
        ]
    )


# ------------------------------------------------------------
# 3.12 Sparse-only frozen fusion diagnostics
# ------------------------------------------------------------

q3_sparse_fusion_df = (
    q3_fusion_df[
        q3_fusion_df[
            "density_stratum"
        ].isin(
            SPARSE_STRATA
        )
    ]
    .copy()
)


assert (
    len(
        q3_sparse_fusion_df
    )
    == 3 * 60
)


# ------------------------------------------------------------
# 3.13 Summary helper
# ------------------------------------------------------------

fusion_summary_records = []


for (
    checkpoint_id,
    density_stratum,
), group in q3_sparse_fusion_df.groupby(
    [
        "checkpoint_id",
        "density_stratum",
    ],
    sort=False,
):

    row0 = group.iloc[
        0
    ]


    total_gt = int(
        group[
            "gt_count"
        ].sum()
    )

    c_pred = int(
        group[
            "c_pred_count"
        ].sum()
    )

    d_pred = int(
        group[
            "d_pred_count"
        ].sum()
    )

    e_pred = int(
        group[
            "e_pred_count"
        ].sum()
    )

    c_tp = int(
        group[
            "c_tp"
        ].sum()
    )

    d_tp = int(
        group[
            "d_tp"
        ].sum()
    )

    e_tp = int(
        group[
            "e_tp"
        ].sum()
    )

    c_fp = int(
        group[
            "c_fp"
        ].sum()
    )

    d_fp = int(
        group[
            "d_fp"
        ].sum()
    )

    e_fp = int(
        group[
            "e_fp"
        ].sum()
    )


    delta_pred_c = (
        e_pred
        - c_pred
    )

    delta_tp_c = (
        e_tp
        - c_tp
    )

    delta_fp_c = (
        e_fp
        - c_fp
    )


    delta_pred_d = (
        e_pred
        - d_pred
    )

    delta_tp_d = (
        e_tp
        - d_tp
    )

    delta_fp_d = (
        e_fp
        - d_fp
    )


    # Diagnostic only:
    # additional FP paid for each additional TP.
    # Only meaningful when delta TP > 0.
    fp_cost_per_extra_tp_vs_c = (
        delta_fp_c
        /
        delta_tp_c
        if delta_tp_c > 0
        else np.nan
    )

    fp_cost_per_extra_tp_vs_d = (
        delta_fp_d
        /
        delta_tp_d
        if delta_tp_d > 0
        else np.nan
    )


    fusion_summary_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "fusion_family":
                row0[
                    "fusion_family"
                ],

            "c_variant":
                row0[
                    "c_variant"
                ],

            "d_variant":
                row0[
                    "d_variant"
                ],

            "tau":
                float(
                    row0[
                        "tau"
                    ]
                ),

            "density_stratum":
                density_stratum,

            "num_images":
                int(
                    len(
                        group
                    )
                ),

            "total_gt":
                total_gt,

            # ----------------------------------------------
            # Parent and fused counts
            # ----------------------------------------------

            "c_pred":
                c_pred,

            "d_pred":
                d_pred,

            "e_pred":
                e_pred,

            "c_tp":
                c_tp,

            "d_tp":
                d_tp,

            "e_tp":
                e_tp,

            "c_fp":
                c_fp,

            "d_fp":
                d_fp,

            "e_fp":
                e_fp,

            # ----------------------------------------------
            # Fused precision / recall
            # ----------------------------------------------

            "e_precision":
                (
                    e_tp
                    /
                    e_pred
                    if e_pred > 0
                    else 0.0
                ),

            "e_recall":
                (
                    e_tp
                    /
                    total_gt
                ),

            # ----------------------------------------------
            # Marginal value relative to C
            # ----------------------------------------------

            "delta_pred_vs_c":
                delta_pred_c,

            "delta_tp_vs_c":
                delta_tp_c,

            "delta_fp_vs_c":
                delta_fp_c,

            "fp_cost_per_extra_tp_vs_c":
                fp_cost_per_extra_tp_vs_c,

            # ----------------------------------------------
            # Marginal value relative to D
            # ----------------------------------------------

            "delta_pred_vs_d":
                delta_pred_d,

            "delta_tp_vs_d":
                delta_tp_d,

            "delta_fp_vs_d":
                delta_fp_d,

            "fp_cost_per_extra_tp_vs_d":
                fp_cost_per_extra_tp_vs_d,
        }
    )


q3_fusion_summary_df = pd.DataFrame(
    fusion_summary_records
)


q3_fusion_summary_df[
    "density_stratum"
] = pd.Categorical(
    q3_fusion_summary_df[
        "density_stratum"
    ],
    categories=SPARSE_STRATA,
    ordered=True,
)


q3_fusion_summary_df = (
    q3_fusion_summary_df
    .sort_values(
        [
            "checkpoint_id",
            "density_stratum",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(
        q3_fusion_summary_df
    )
    == 9
)


# ------------------------------------------------------------
# 3.14 Attach E2 oracle complementarity for exact parent pairs
# ------------------------------------------------------------

e2_exact_pair_rows = []


for checkpoint_id, cfg in FROZEN_FUSIONS.items():

    pair = (
        q3_e2_summary_df[
            (
                q3_e2_summary_df[
                    "c_variant"
                ]
                == cfg[
                    "c_variant"
                ]
            )
            &
            (
                q3_e2_summary_df[
                    "d_variant"
                ]
                == cfg[
                    "d_variant"
                ]
            )
        ]
        .copy()
    )


    assert (
        len(
            pair
        )
        == 3
    )


    pair[
        "checkpoint_id"
    ] = checkpoint_id


    e2_exact_pair_rows.append(
        pair
    )


q3_e2_exact_pairs_df = (
    pd.concat(
        e2_exact_pair_rows,
        ignore_index=True,
    )
)


q3_fusion_summary_df = (
    q3_fusion_summary_df
    .merge(
        q3_e2_exact_pairs_df[
            [
                "checkpoint_id",
                "density_stratum",
                "both",
                "c_only",
                "d_only",
                "neither",
                "oracle_union_tp",
                "oracle_union_recall",
                "d_only_rate_of_gt",
                "d_unique_fraction_of_d_coverage",
            ]
        ],
        on=[
            "checkpoint_id",
            "density_stratum",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 3.15 How much of the oracle C∪D opportunity does E recover?
# ------------------------------------------------------------

q3_fusion_summary_df[
    "fusion_fraction_of_oracle_union_tp"
] = (
    q3_fusion_summary_df[
        "e_tp"
    ]
    /
    q3_fusion_summary_df[
        "oracle_union_tp"
    ]
)


# Additional TP beyond C that are theoretically available
# from D-only GT coverage.

q3_fusion_summary_df[
    "available_d_only_gt"
] = (
    q3_fusion_summary_df[
        "d_only"
    ]
)


q3_fusion_summary_df[
    "fraction_available_d_only_recovered_by_fusion"
] = np.where(
    q3_fusion_summary_df[
        "available_d_only_gt"
    ]
    > 0,

    np.maximum(
        q3_fusion_summary_df[
            "delta_tp_vs_c"
        ],
        0,
    )
    /
    q3_fusion_summary_df[
        "available_d_only_gt"
    ],

    np.nan,
)


# ------------------------------------------------------------
# 3.16 Save outputs
# ------------------------------------------------------------

q3_e2_summary_df.to_csv(
    Q3_E2_COMPLEMENTARITY_FILE,
    index=False,
)

q3_sparse_fusion_df.to_csv(
    Q3_FROZEN_FUSION_PER_IMAGE_FILE,
    index=False,
)

q3_fusion_summary_df.to_csv(
    Q3_FROZEN_FUSION_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 3.17 Compact displays
# ------------------------------------------------------------

E2_DISPLAY_COLS = [
    "c_variant",
    "d_variant",
    "density_stratum",
    "total_gt",
    "both",
    "c_only",
    "d_only",
    "neither",
    "c_recall_from_e2",
    "d_recall_from_e2",
    "oracle_union_recall",
    "d_only_rate_of_gt",
    "d_unique_fraction_of_d_coverage",
]


e2_display_df = (
    q3_e2_summary_df[
        E2_DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "c_recall_from_e2",
    "d_recall_from_e2",
    "oracle_union_recall",
    "d_only_rate_of_gt",
    "d_unique_fraction_of_d_coverage",
]:

    e2_display_df[
        col
    ] = (
        e2_display_df[
            col
        ]
        .round(
            4
        )
    )


FUSION_DISPLAY_COLS = [
    "checkpoint_id",
    "fusion_family",
    "c_variant",
    "d_variant",
    "tau",
    "density_stratum",
    "total_gt",
    "c_pred",
    "d_pred",
    "e_pred",
    "c_tp",
    "d_tp",
    "e_tp",
    "c_fp",
    "d_fp",
    "e_fp",
    "delta_pred_vs_c",
    "delta_tp_vs_c",
    "delta_fp_vs_c",
    "fp_cost_per_extra_tp_vs_c",
    "d_only",
    "oracle_union_tp",
    "fusion_fraction_of_oracle_union_tp",
    "fraction_available_d_only_recovered_by_fusion",
]


fusion_display_df = (
    q3_fusion_summary_df[
        FUSION_DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "tau",
    "fp_cost_per_extra_tp_vs_c",
    "fusion_fraction_of_oracle_union_tp",
    "fraction_available_d_only_recovered_by_fusion",
]:

    fusion_display_df[
        col
    ] = (
        fusion_display_df[
            col
        ]
        .round(
            4
        )
    )


# ------------------------------------------------------------
# 3.18 Pivots for central questions
# ------------------------------------------------------------

d_only_pivot = (
    q3_e2_exact_pairs_df
    .pivot(
        index="checkpoint_id",
        columns="density_stratum",
        values="d_only_rate_of_gt",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


fusion_oracle_pivot = (
    q3_fusion_summary_df
    .pivot(
        index="checkpoint_id",
        columns="density_stratum",
        values="fusion_fraction_of_oracle_union_tp",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


fp_cost_pivot = (
    q3_fusion_summary_df
    .pivot(
        index="checkpoint_id",
        columns="density_stratum",
        values="fp_cost_per_extra_tp_vs_c",
    )
    .reindex(
        columns=SPARSE_STRATA
    )
)


# ------------------------------------------------------------
# 3.19 Confirmation
# ------------------------------------------------------------

print("=" * 125)
print("STEP 3 - Q3: FUSION MARGINAL-VALUE ANALYSIS")
print("=" * 125)

print(
    "E2 sparse rows               :",
    len(
        e2_sparse_df
    ),
    "/ 900"
)

print(
    "Frozen E3 rows               :",
    len(
        frozen_e3_df
    ),
    "/ 300"
)

print(
    "Frozen sparse E3 rows        :",
    len(
        q3_sparse_fusion_df
    ),
    "/ 180"
)

print(
    "Exact parent joins           : PASS"
)

print(
    "E2 oracle-union identity     : PASS"
)

print(
    "New inference                : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nAll C/D pair complementarity in S1-S3:"
)

display(
    e2_display_df
)


print(
    "\nFrozen fusion representatives - marginal value:"
)

display(
    fusion_display_df
)


print(
    "\nD-only GT opportunity for the frozen parent pairs:"
)

display(
    d_only_pivot
    .round(
        3
    )
)


print(
    "\nFraction of oracle C∪D TP actually recovered by fusion:"
)

display(
    fusion_oracle_pivot
    .round(
        3
    )
)


print(
    "\nAdditional FP cost per additional TP relative to C:"
)

display(
    fp_cost_pivot
    .round(
        3
    )
)


print(
    "\nSaved E2 complementarity:",
    Q3_E2_COMPLEMENTARITY_FILE,
)

print(
    "Saved per-image frozen fusion diagnostics:",
    Q3_FROZEN_FUSION_PER_IMAGE_FILE,
)

print(
    "Saved frozen fusion summary:",
    Q3_FROZEN_FUSION_SUMMARY_FILE,
)

print("=" * 125)

STEP 3 - Q3: FUSION MARGINAL-VALUE ANALYSIS
E2 sparse rows               : 900 / 900
Frozen E3 rows               : 300 / 300
Frozen sparse E3 rows        : 180 / 180
Exact parent joins           : PASS
E2 oracle-union identity     : PASS
New inference                : NO
Threshold tuning             : NO
Test set accessed            : NO

All C/D pair complementarity in S1-S3:


,c_variant,d_variant,density_stratum,total_gt,both,c_only,d_only,neither,c_recall_from_e2,d_recall_from_e2,oracle_union_recall,d_only_rate_of_gt,d_unique_fraction_of_d_coverage
0,C1,D1,S1_0_20,195,33,30,56,76,0.3231,0.4564,0.6103,0.2872,0.6292
1,C1,D1,S2_20_40,320,77,17,94,132,0.2938,0.5344,0.5875,0.2938,0.5497
2,C1,D1,S3_40_60,537,70,40,183,244,0.2048,0.4711,0.5456,0.3408,0.7233
3,C1,D4,S1_0_20,195,33,30,44,88,0.3231,0.3949,0.5487,0.2256,0.5714
4,C1,D4,S2_20_40,320,76,18,66,160,0.2938,0.4438,0.5000,0.2062,0.4648
5,C1,D4,S3_40_60,537,70,40,137,290,0.2048,0.3855,0.4600,0.2551,0.6618
6,C1,D5-768,S1_0_20,195,33,30,68,64,0.3231,0.5179,0.6718,0.3487,0.6733
7,C1,D5-768,S2_20_40,320,70,24,139,87,0.2938,0.6531,0.7281,0.4344,0.6651
8,C1,D5-768,S3_40_60,537,70,40,236,191,0.2048,0.5698,0.6443,0.4395,0.7712
9,C1,D6B-T-native,S1_0_20,195,46,17,87,45,0.3231,0.6821,0.7692,0.4462,0.6541



Frozen fusion representatives - marginal value:


,checkpoint_id,fusion_family,c_variant,d_variant,tau,density_stratum,total_gt,c_pred,d_pred,e_pred,...,d_fp,e_fp,delta_pred_vs_c,delta_tp_vs_c,delta_fp_vs_c,fp_cost_per_extra_tp_vs_c,d_only,oracle_union_tp,fusion_fraction_of_oracle_union_tp,fraction_available_d_only_recovered_by_fusion
0,dense_recovery,F1,C1,D6B9-strict,0.050,S1_0_20,195,240,529,627,...,433,524,387,40,347,8.6750,58,121,0.8512,0.6897
1,dense_recovery,F1,C1,D6B9-strict,0.050,S2_20_40,320,392,852,1001,...,670,805,609,102,507,4.9706,118,212,0.9245,0.8644
2,dense_recovery,F1,C1,D6B9-strict,0.050,S3_40_60,537,401,1017,1186,...,808,953,785,123,662,5.3821,147,257,0.9066,0.8367
3,overall_joint,F2,C2b,D6B-T-native,0.010,S1_0_20,195,759,2024,827,...,1891,721,68,11,57,5.1818,59,154,0.6883,0.1864
4,overall_joint,F2,C2b,D6B-T-native,0.010,S2_20_40,320,1170,2850,972,...,2600,805,-198,6,-204,-34.0000,115,276,0.6051,0.0522
5,overall_joint,F2,C2b,D6B-T-native,0.010,S3_40_60,537,1224,2972,913,...,2613,671,-311,-4,-307,NaN,176,422,0.5735,0.0000
6,sparse_joint,F2,C2b,D6B-T-native,0.005,S1_0_20,195,759,2024,753,...,1891,651,-6,7,-13,-1.8571,59,154,0.6623,0.1186
7,sparse_joint,F2,C2b,D6B-T-native,0.005,S2_20_40,320,1170,2850,877,...,2600,713,-293,3,-296,-98.6667,115,276,0.5942,0.0261
8,sparse_joint,F2,C2b,D6B-T-native,0.005,S3_40_60,537,1224,2972,804,...,2613,577,-420,-19,-401,NaN,176,422,0.5379,0.0000



D-only GT opportunity for the frozen parent pairs:


...


Fraction of oracle C∪D TP actually recovered by fusion:


density_stratum,S1_0_20,S2_20_40,S3_40_60
checkpoint_id,,,
dense_recovery,0.851,0.925,0.907
overall_joint,0.688,0.605,0.573
sparse_joint,0.662,0.594,0.538



Additional FP cost per additional TP relative to C:


density_stratum,S1_0_20,S2_20_40,S3_40_60
checkpoint_id,,,
dense_recovery,8.675,4.971,5.382
overall_joint,5.182,-34.000,NaN
sparse_joint,-1.857,-98.667,NaN



Saved E2 complementarity: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q3_E2_complementarity_sparse.csv
Saved per-image frozen fusion diagnostics: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q3_frozen_fusion_marginal_value_per_image.csv
Saved frozen fusion summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q3_frozen_fusion_marginal_value_summary.csv


In [6]:
# ============================================================
# Step 4 - Q4: Help-vs-Hurt Image Characteristics
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 4.1 Output files
# ------------------------------------------------------------

Q4_PER_IMAGE_FILE = (
    DIAG_TABLES_DIR
    / "Q4_help_vs_hurt_per_image_S1_S3.csv"
)

Q4_OUTCOME_COUNTS_FILE = (
    DIAG_TABLES_DIR
    / "Q4_help_vs_hurt_outcome_counts.csv"
)

Q4_SUMMARY_FILE = (
    DIAG_TABLES_DIR
    / "Q4_help_vs_hurt_characteristics_summary.csv"
)

Q4_EXTREME_CASES_FILE = (
    DIAG_TABLES_DIR
    / "Q4_help_vs_hurt_extreme_cases.csv"
)


# ------------------------------------------------------------
# 4.2 Build frozen B0 per-image reference
# ------------------------------------------------------------

B0_REQUIRED = {
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "pred_count",
}


missing_b0 = (
    B0_REQUIRED
    - set(
        b0_df.columns
    )
)


assert (
    len(
        missing_b0
    )
    == 0
), (
    f"B0 missing required columns: "
    f"{sorted(missing_b0)}"
)


b0_q4_df = (
    b0_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "pred_count",
        ]
    ]
    .copy()
    .rename(
        columns={
            "pred_count":
                "b0_pred_count",
        }
    )
)


b0_q4_df[
    "b0_signed_error"
] = (
    b0_q4_df[
        "b0_pred_count"
    ]
    -
    b0_q4_df[
        "gt_count"
    ]
)


b0_q4_df[
    "b0_abs_error"
] = (
    b0_q4_df[
        "b0_signed_error"
    ]
    .abs()
)


b0_q4_df[
    "b0_pred_gt_ratio"
] = (
    b0_q4_df[
        "b0_pred_count"
    ]
    /
    b0_q4_df[
        "gt_count"
    ]
)


# ------------------------------------------------------------
# 4.3 Start from exact frozen fusion rows from Step 3
# ------------------------------------------------------------

q4_df = (
    q3_sparse_fusion_df
    .copy()
)


assert (
    len(
        q4_df
    )
    == 180
)


# ------------------------------------------------------------
# 4.4 Attach B0
# ------------------------------------------------------------

q4_df = (
    q4_df
    .merge(
        b0_q4_df,
        on=[
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    q4_df[
        "b0_pred_count"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 4.5 Guided count errors
#
# Under the frozen protocol:
# E3 fused point count == guided SAM2 retained-mask count.
# ------------------------------------------------------------

q4_df[
    "guided_signed_error"
] = (
    q4_df[
        "e_pred_count"
    ]
    -
    q4_df[
        "gt_count"
    ]
)


q4_df[
    "guided_abs_error"
] = (
    q4_df[
        "guided_signed_error"
    ]
    .abs()
)


q4_df[
    "guided_pred_gt_ratio"
] = (
    q4_df[
        "e_pred_count"
    ]
    /
    q4_df[
        "gt_count"
    ]
)


# Negative = guided improved relative to B0
# Positive = guided worsened relative to B0

q4_df[
    "delta_abs_error_vs_b0"
] = (
    q4_df[
        "guided_abs_error"
    ]
    -
    q4_df[
        "b0_abs_error"
    ]
)


q4_df[
    "outcome_vs_b0"
] = np.select(
    [
        q4_df[
            "delta_abs_error_vs_b0"
        ]
        < 0,

        q4_df[
            "delta_abs_error_vs_b0"
        ]
        > 0,
    ],
    [
        "improved",
        "worsened",
    ],
    default="same",
)


# ------------------------------------------------------------
# 4.6 Parent C/D image-level metrics
# ------------------------------------------------------------

for parent in [
    "c",
    "d",
]:

    q4_df[
        f"{parent}_pred_gt_ratio"
    ] = (
        q4_df[
            f"{parent}_pred_count"
        ]
        /
        q4_df[
            "gt_count"
        ]
    )


    q4_df[
        f"{parent}_precision"
    ] = np.where(
        q4_df[
            f"{parent}_pred_count"
        ]
        > 0,

        q4_df[
            f"{parent}_tp"
        ]
        /
        q4_df[
            f"{parent}_pred_count"
        ],

        0.0,
    )


    q4_df[
        f"{parent}_recall"
    ] = (
        q4_df[
            f"{parent}_tp"
        ]
        /
        q4_df[
            "gt_count"
        ]
    )


# Fused precision / recall

q4_df[
    "e_precision"
] = np.where(
    q4_df[
        "e_pred_count"
    ]
    > 0,

    q4_df[
        "e_tp"
    ]
    /
    q4_df[
        "e_pred_count"
    ],

    0.0,
)


q4_df[
    "e_recall"
] = (
    q4_df[
        "e_tp"
    ]
    /
    q4_df[
        "gt_count"
    ]
)


# ------------------------------------------------------------
# 4.7 Attach E2 exact-pair complementarity per image
# ------------------------------------------------------------

e2_q4_parts = []


for checkpoint_id, cfg in FROZEN_FUSIONS.items():

    part = (
        e2_sparse_df[
            (
                e2_sparse_df[
                    "c_variant"
                ]
                == cfg[
                    "c_variant"
                ]
            )
            &
            (
                e2_sparse_df[
                    "d_variant"
                ]
                == cfg[
                    "d_variant"
                ]
            )
        ]
        .copy()
    )


    assert (
        len(
            part
        )
        == 60
    )


    part[
        "checkpoint_id"
    ] = checkpoint_id


    e2_q4_parts.append(
        part
    )


e2_q4_df = (
    pd.concat(
        e2_q4_parts,
        ignore_index=True,
    )
)


assert (
    len(
        e2_q4_df
    )
    == 180
)


E2_Q4_KEEP = [
    "checkpoint_id",
    "dataset_index",
    "image_id",
    "density_stratum",
    "c_variant",
    "d_variant",
    "both",
    "c_only",
    "d_only",
    "neither",
    "oracle_union_tp",
]


q4_df = (
    q4_df
    .merge(
        e2_q4_df[
            E2_Q4_KEEP
        ],
        on=[
            "checkpoint_id",
            "dataset_index",
            "image_id",
            "density_stratum",
            "c_variant",
            "d_variant",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    q4_df[
        "d_only"
    ]
    .notna()
    .all()
)


q4_df[
    "d_only_rate_of_gt"
] = (
    q4_df[
        "d_only"
    ]
    /
    q4_df[
        "gt_count"
    ]
)


q4_df[
    "oracle_union_recall"
] = (
    q4_df[
        "oracle_union_tp"
    ]
    /
    q4_df[
        "gt_count"
    ]
)


# ------------------------------------------------------------
# 4.8 Attach Step-2 spatial diagnostics for exact C/D parents
# ------------------------------------------------------------

Q2_SPATIAL_KEEP = [
    "dataset_index",
    "image_id",
    "density_stratum",
    "candidate_id",
    "rate_pred_within_1R",
    "rate_pred_between_1R_2R",
    "rate_pred_beyond_2R",
    "rate_local_surplus_within_1R",
    "gt_zero_local_rate",
    "gt_multiple_local_rate",
]


# -------------------------
# C parent spatial features
# -------------------------

c_spatial_df = (
    q2_image_df[
        q2_image_df[
            "branch"
        ]
        == "C"
    ][
        Q2_SPATIAL_KEEP
    ]
    .rename(
        columns={
            "candidate_id":
                "c_variant",

            "rate_pred_within_1R":
                "c_rate_within_1R",

            "rate_pred_between_1R_2R":
                "c_rate_1R_2R",

            "rate_pred_beyond_2R":
                "c_remote_rate",

            "rate_local_surplus_within_1R":
                "c_local_surplus_rate",

            "gt_zero_local_rate":
                "c_gt_zero_local_rate",

            "gt_multiple_local_rate":
                "c_gt_multiple_local_rate",
        }
    )
)


# -------------------------
# D parent spatial features
# -------------------------

d_spatial_df = (
    q2_image_df[
        q2_image_df[
            "branch"
        ]
        == "D"
    ][
        Q2_SPATIAL_KEEP
    ]
    .rename(
        columns={
            "candidate_id":
                "d_variant",

            "rate_pred_within_1R":
                "d_rate_within_1R",

            "rate_pred_between_1R_2R":
                "d_rate_1R_2R",

            "rate_pred_beyond_2R":
                "d_remote_rate",

            "rate_local_surplus_within_1R":
                "d_local_surplus_rate",

            "gt_zero_local_rate":
                "d_gt_zero_local_rate",

            "gt_multiple_local_rate":
                "d_gt_multiple_local_rate",
        }
    )
)


q4_df = (
    q4_df
    .merge(
        c_spatial_df,
        on=[
            "dataset_index",
            "image_id",
            "density_stratum",
            "c_variant",
        ],
        how="left",
        validate="many_to_one",
    )
    .merge(
        d_spatial_df,
        on=[
            "dataset_index",
            "image_id",
            "density_stratum",
            "d_variant",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    q4_df[
        [
            "c_remote_rate",
            "d_remote_rate",
        ]
    ]
    .notna()
    .all()
    .all()
)


# ------------------------------------------------------------
# 4.9 Fusion change relative to C
# ------------------------------------------------------------

q4_df[
    "fusion_pred_change_vs_c"
] = (
    q4_df[
        "e_pred_count"
    ]
    -
    q4_df[
        "c_pred_count"
    ]
)


q4_df[
    "fusion_tp_change_vs_c"
] = (
    q4_df[
        "e_tp"
    ]
    -
    q4_df[
        "c_tp"
    ]
)


q4_df[
    "fusion_fp_change_vs_c"
] = (
    q4_df[
        "e_fp"
    ]
    -
    q4_df[
        "c_fp"
    ]
)


# ------------------------------------------------------------
# 4.10 Frozen consistency check against known sparse outcomes
#
# From the end-to-end checkpoint:
# overall_joint : 14 improved? no assumption here.
# We deliberately reconstruct directly from the frozen counts.
# ------------------------------------------------------------

assert (
    q4_df[
        "checkpoint_id"
    ]
    .nunique()
    == 3
)

assert (
    q4_df.groupby(
        "checkpoint_id"
    ).size()
    .eq(
        60
    )
    .all()
)


# ------------------------------------------------------------
# 4.11 Outcome counts
# ------------------------------------------------------------

OUTCOME_ORDER = [
    "improved",
    "same",
    "worsened",
]


q4_outcome_counts_df = (
    q4_df
    .groupby(
        [
            "checkpoint_id",
            "outcome_vs_b0",
        ],
        observed=False,
    )
    .size()
    .rename(
        "num_images"
    )
    .reset_index()
)


# Add proportions explicitly

checkpoint_totals = (
    q4_outcome_counts_df
    .groupby(
        "checkpoint_id"
    )[
        "num_images"
    ]
    .transform(
        "sum"
    )
)


q4_outcome_counts_df[
    "fraction_images"
] = (
    q4_outcome_counts_df[
        "num_images"
    ]
    /
    checkpoint_totals
)


# ------------------------------------------------------------
# 4.12 Characteristics to compare
# ------------------------------------------------------------

Q4_FEATURES = [
    # Scene / baseline
    "gt_count",
    "b0_pred_count",
    "b0_pred_gt_ratio",
    "b0_signed_error",
    "b0_abs_error",

    # C parent
    "c_pred_gt_ratio",
    "c_precision",
    "c_recall",
    "c_remote_rate",
    "c_gt_zero_local_rate",

    # D parent
    "d_pred_gt_ratio",
    "d_precision",
    "d_recall",
    "d_remote_rate",
    "d_gt_zero_local_rate",

    # C/D complementarity
    "d_only_rate_of_gt",
    "oracle_union_recall",

    # Actual fusion
    "guided_pred_gt_ratio",
    "e_precision",
    "e_recall",
    "fusion_pred_change_vs_c",
    "fusion_tp_change_vs_c",
    "fusion_fp_change_vs_c",
]


# ------------------------------------------------------------
# 4.13 Summary by checkpoint and help/hurt outcome
#
# Use both mean and median.
# Median is especially useful because some images have
# extreme point/count values.
# ------------------------------------------------------------

summary_records = []


for (
    checkpoint_id,
    outcome,
), group in q4_df.groupby(
    [
        "checkpoint_id",
        "outcome_vs_b0",
    ],
    sort=False,
):

    record = {
        "checkpoint_id":
            checkpoint_id,

        "outcome_vs_b0":
            outcome,

        "num_images":
            int(
                len(
                    group
                )
            ),
    }


    for feature in Q4_FEATURES:

        record[
            f"{feature}__mean"
        ] = float(
            group[
                feature
            ].mean()
        )

        record[
            f"{feature}__median"
        ] = float(
            group[
                feature
            ].median()
        )


    summary_records.append(
        record
    )


q4_summary_df = pd.DataFrame(
    summary_records
)


# ------------------------------------------------------------
# 4.14 Compact diagnostic summary
#
# Keep the most interpretable features visible.
# ------------------------------------------------------------

COMPACT_FEATURES = [
    "gt_count",
    "b0_abs_error",
    "b0_pred_gt_ratio",
    "c_pred_gt_ratio",
    "c_precision",
    "c_recall",
    "d_pred_gt_ratio",
    "d_precision",
    "d_recall",
    "d_remote_rate",
    "d_only_rate_of_gt",
    "guided_pred_gt_ratio",
    "e_precision",
    "e_recall",
    "fusion_pred_change_vs_c",
    "fusion_tp_change_vs_c",
    "fusion_fp_change_vs_c",
]


compact_cols = [
    "checkpoint_id",
    "outcome_vs_b0",
    "num_images",
]


for feature in COMPACT_FEATURES:

    compact_cols.extend(
        [
            f"{feature}__mean",
            f"{feature}__median",
        ]
    )


q4_compact_summary_df = (
    q4_summary_df[
        compact_cols
    ]
    .copy()
)


# Round floats only for display

for col in q4_compact_summary_df.columns:

    if (
        col
        not in [
            "checkpoint_id",
            "outcome_vs_b0",
            "num_images",
        ]
    ):

        q4_compact_summary_df[
            col
        ] = (
            q4_compact_summary_df[
                col
            ]
            .round(
                4
            )
        )


# ------------------------------------------------------------
# 4.15 A more readable median-only table
#
# This will probably be the most useful table for interpretation.
# ------------------------------------------------------------

MEDIAN_DISPLAY_FEATURES = [
    "gt_count",
    "b0_abs_error",
    "b0_pred_gt_ratio",
    "c_pred_gt_ratio",
    "d_pred_gt_ratio",
    "d_remote_rate",
    "d_only_rate_of_gt",
    "guided_pred_gt_ratio",
    "e_precision",
    "e_recall",
    "fusion_pred_change_vs_c",
]


median_display_cols = [
    "checkpoint_id",
    "outcome_vs_b0",
    "num_images",
] + [
    f"{feature}__median"
    for feature in MEDIAN_DISPLAY_FEATURES
]


q4_median_display_df = (
    q4_summary_df[
        median_display_cols
    ]
    .copy()
)


for col in q4_median_display_df.columns:

    if col.endswith(
        "__median"
    ):

        q4_median_display_df[
            col
        ] = (
            q4_median_display_df[
                col
            ]
            .round(
                3
            )
        )


# ------------------------------------------------------------
# 4.16 Extreme help/hurt examples
#
# Diagnostic only: identify cases for later visual inspection.
# ------------------------------------------------------------

extreme_parts = []


for checkpoint_id, group in q4_df.groupby(
    "checkpoint_id"
):

    improved = (
        group[
            group[
                "delta_abs_error_vs_b0"
            ]
            < 0
        ]
        .sort_values(
            "delta_abs_error_vs_b0",
            ascending=True,
        )
        .head(
            5
        )
        .copy()
    )


    worsened = (
        group[
            group[
                "delta_abs_error_vs_b0"
            ]
            > 0
        ]
        .sort_values(
            "delta_abs_error_vs_b0",
            ascending=False,
        )
        .head(
            5
        )
        .copy()
    )


    improved[
        "extreme_type"
    ] = "largest_improvement"

    worsened[
        "extreme_type"
    ] = "largest_worsening"


    improved[
        "rank"
    ] = np.arange(
        1,
        len(
            improved
        )
        + 1,
    )

    worsened[
        "rank"
    ] = np.arange(
        1,
        len(
            worsened
        )
        + 1,
    )


    extreme_parts.extend(
        [
            improved,
            worsened,
        ]
    )


q4_extreme_df = (
    pd.concat(
        extreme_parts,
        ignore_index=True,
    )
)


EXTREME_DISPLAY_COLS = [
    "checkpoint_id",
    "extreme_type",
    "rank",
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "b0_pred_count",
    "e_pred_count",
    "b0_abs_error",
    "guided_abs_error",
    "delta_abs_error_vs_b0",
    "c_pred_gt_ratio",
    "d_pred_gt_ratio",
    "d_remote_rate",
    "d_only_rate_of_gt",
    "e_precision",
    "e_recall",
]


q4_extreme_display_df = (
    q4_extreme_df[
        EXTREME_DISPLAY_COLS
    ]
    .copy()
)


for col in [
    "c_pred_gt_ratio",
    "d_pred_gt_ratio",
    "d_remote_rate",
    "d_only_rate_of_gt",
    "e_precision",
    "e_recall",
]:

    q4_extreme_display_df[
        col
    ] = (
        q4_extreme_display_df[
            col
        ]
        .round(
            4
        )
    )


# ------------------------------------------------------------
# 4.17 Useful pivots
# ------------------------------------------------------------

outcome_pivot = (
    q4_outcome_counts_df
    .pivot(
        index="checkpoint_id",
        columns="outcome_vs_b0",
        values="num_images",
    )
    .reindex(
        columns=OUTCOME_ORDER
    )
    .fillna(
        0
    )
    .astype(
        int
    )
)


b0_error_median_pivot = (
    q4_summary_df
    .pivot(
        index="checkpoint_id",
        columns="outcome_vs_b0",
        values="b0_abs_error__median",
    )
    .reindex(
        columns=OUTCOME_ORDER
    )
)


d_remote_median_pivot = (
    q4_summary_df
    .pivot(
        index="checkpoint_id",
        columns="outcome_vs_b0",
        values="d_remote_rate__median",
    )
    .reindex(
        columns=OUTCOME_ORDER
    )
)


d_only_median_pivot = (
    q4_summary_df
    .pivot(
        index="checkpoint_id",
        columns="outcome_vs_b0",
        values="d_only_rate_of_gt__median",
    )
    .reindex(
        columns=OUTCOME_ORDER
    )
)


guided_ratio_median_pivot = (
    q4_summary_df
    .pivot(
        index="checkpoint_id",
        columns="outcome_vs_b0",
        values="guided_pred_gt_ratio__median",
    )
    .reindex(
        columns=OUTCOME_ORDER
    )
)


# ------------------------------------------------------------
# 4.18 Save outputs
# ------------------------------------------------------------

q4_df.to_csv(
    Q4_PER_IMAGE_FILE,
    index=False,
)

q4_outcome_counts_df.to_csv(
    Q4_OUTCOME_COUNTS_FILE,
    index=False,
)

q4_summary_df.to_csv(
    Q4_SUMMARY_FILE,
    index=False,
)

q4_extreme_df.to_csv(
    Q4_EXTREME_CASES_FILE,
    index=False,
)


# ------------------------------------------------------------
# 4.19 Confirmation + displays
# ------------------------------------------------------------

print("=" * 125)
print("STEP 4 - Q4: HELP-vs-HURT IMAGE CHARACTERISTICS IN S1-S3")
print("=" * 125)

print(
    "Image-system rows            :",
    len(
        q4_df
    ),
    "/ 180"
)

print(
    "Systems                      :",
    sorted(
        q4_df[
            "checkpoint_id"
        ]
        .unique()
        .tolist()
    )
)

print(
    "Images per system            : 60"
)

print(
    "Exact B0 join                : PASS"
)

print(
    "Exact E2 parent-pair join    : PASS"
)

print(
    "Exact Q2 spatial joins       : PASS"
)

print(
    "New inference                : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nHelp / same / hurt counts:"
)

display(
    outcome_pivot
)


print(
    "\nMedian diagnostic characteristics:"
)

display(
    q4_median_display_df
)


print(
    "\nMedian B0 absolute error by outcome:"
)

display(
    b0_error_median_pivot
    .round(
        3
    )
)


print(
    "\nMedian D remote-point rate (>2R) by outcome:"
)

display(
    d_remote_median_pivot
    .round(
        3
    )
)


print(
    "\nMedian D-only GT opportunity by outcome:"
)

display(
    d_only_median_pivot
    .round(
        3
    )
)


print(
    "\nMedian final guided Pred/GT ratio by outcome:"
)

display(
    guided_ratio_median_pivot
    .round(
        3
    )
)


print(
    "\nFive largest improvements and worsenings per system:"
)

display(
    q4_extreme_display_df
)


print(
    "\nSaved per-image Q4 diagnostics:",
    Q4_PER_IMAGE_FILE,
)

print(
    "Saved outcome counts:",
    Q4_OUTCOME_COUNTS_FILE,
)

print(
    "Saved characteristics summary:",
    Q4_SUMMARY_FILE,
)

print(
    "Saved extreme cases:",
    Q4_EXTREME_CASES_FILE,
)

print("=" * 125)

STEP 4 - Q4: HELP-vs-HURT IMAGE CHARACTERISTICS IN S1-S3
Image-system rows            : 180 / 180
Systems                      : ['dense_recovery', 'overall_joint', 'sparse_joint']
Images per system            : 60
Exact B0 join                : PASS
Exact E2 parent-pair join    : PASS
Exact Q2 spatial joins       : PASS
New inference                : NO
Threshold tuning             : NO
Test set accessed            : NO

Help / same / hurt counts:


outcome_vs_b0,improved,same,worsened
checkpoint_id,,,
dense_recovery,18,0,42
overall_joint,14,0,46
sparse_joint,14,0,46



Median diagnostic characteristics:


,checkpoint_id,outcome_vs_b0,num_images,gt_count__median,b0_abs_error__median,b0_pred_gt_ratio__median,c_pred_gt_ratio__median,d_pred_gt_ratio__median,d_remote_rate__median,d_only_rate_of_gt__median,guided_pred_gt_ratio__median,e_precision__median,e_recall__median,fusion_pred_change_vs_c__median
0,overall_joint,improved,14,22.5,13.5,0.083,2.943,1.190,0.076,0.111,1.149,0.566,0.655,-33.0
1,overall_joint,worsened,46,15.0,5.0,0.829,2.661,11.000,0.691,0.406,2.955,0.121,0.444,6.0
2,sparse_joint,improved,14,22.5,13.5,0.083,2.943,1.190,0.076,0.111,1.113,0.596,0.640,-36.5
3,sparse_joint,worsened,46,15.0,5.0,0.829,2.661,11.000,0.691,0.406,2.778,0.133,0.413,-2.0
4,dense_recovery,improved,18,21.0,13.0,0.113,0.703,0.646,0.000,0.268,1.022,0.494,0.489,5.5
5,dense_recovery,worsened,42,15.0,5.0,0.866,1.244,2.804,0.570,0.263,3.048,0.156,0.500,35.0



Median B0 absolute error by outcome:


...


Median final guided Pred/GT ratio by outcome:


outcome_vs_b0,improved,same,worsened
checkpoint_id,,,
dense_recovery,1.022,NaN,3.048
overall_joint,1.149,NaN,2.955
sparse_joint,1.113,NaN,2.778



Five largest improvements and worsenings per system:


,checkpoint_id,extreme_type,rank,dataset_index,image_id,category,density_stratum,gt_count,b0_pred_count,e_pred_count,b0_abs_error,guided_abs_error,delta_abs_error_vs_b0,c_pred_gt_ratio,d_pred_gt_ratio,d_remote_rate,d_only_rate_of_gt,e_precision,e_recall
0,dense_recovery,largest_improvement,1,618,4544.jpg,flamingos,S3_40_60,33,1,29,32,4,-28,0.8182,0.1818,0.1667,0.0303,0.5172,0.4545
1,dense_recovery,largest_improvement,2,1183,7305.jpg,birds,S3_40_60,31,2,33,29,2,-27,1.0000,0.1290,0.0000,0.0645,0.2121,0.2258
2,dense_recovery,largest_improvement,3,609,4535.jpg,flamingos,S3_40_60,22,0,25,22,3,-19,0.4091,0.9091,0.0500,0.1818,0.3200,0.3636
3,dense_recovery,largest_improvement,4,650,4578.jpg,flamingos,S3_40_60,24,2,19,22,5,-17,0.4167,0.6250,0.0000,0.4167,0.6316,0.5000
4,dense_recovery,largest_improvement,5,394,2840.jpg,chairs,S3_40_60,35,5,19,30,16,-14,0.0857,0.4857,0.0000,0.2857,0.5263,0.2857
5,dense_recovery,largest_worsening,1,1260,7100.jpg,milk cartons,S3_40_60,27,40,185,13,158,145,0.5556,6.7037,0.7293,0.2593,0.0486,0.3333
6,dense_recovery,largest_worsening,2,488,3543.jpg,chicken wings,S3_40_60,30,28,157,2,127,125,1.3333,4.8667,0.6027,0.5667,0.1146,0.6000
7,dense_recovery,largest_worsening,3,330,1973.jpg,bottle caps,S1_0_20,12,19,108,7,96,89,1.2500,8.7500,0.6286,0.4167,0.0833,0.7500
8,dense_recovery,largest_worsening,4,845,5101.jpg,chicken wings,S3_40_60,23,24,106,1,83,82,1.3043,4.2609,0.6531,0.5652,0.1792,0.8261
9,dense_recovery,largest_worsening,5,835,5091.jpg,chicken wings,S2_20_40,15,20,102,5,87,82,2.9333,5.6000,0.6667,0.4000,0.0588,0.4000



Saved per-image Q4 diagnostics: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q4_help_vs_hurt_per_image_S1_S3.csv
Saved outcome counts: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q4_help_vs_hurt_outcome_counts.csv
Saved characteristics summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q4_help_vs_hurt_characteristics_summary.csv
Saved extreme cases: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Diagnostic/tables/Q4_help_vs_hurt_extreme_cases.csv


In [7]:
# ============================================================
# Step 5 - Diagnostic Synthesis / Working Hypotheses
# ============================================================

import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 5.1 Output files
# ------------------------------------------------------------

Q5_EVIDENCE_FILE = (
    DIAG_TABLES_DIR
    / "Q5_diagnostic_evidence_summary.csv"
)

Q5_DECISION_FILE = (
    DIAG_TABLES_DIR
    / "Q5_architecture_decision_summary.csv"
)

Q5_MANIFEST_FILE = (
    DIAG_METADATA_DIR
    / "Q5_diagnostic_synthesis.json"
)


# ------------------------------------------------------------
# 5.2 Helper functions
# ------------------------------------------------------------

def get_q1_variant(
    branch,
    candidate_id,
):

    row = (
        q1_overall_sparse_df[
            (
                q1_overall_sparse_df[
                    "branch"
                ]
                == branch
            )
            &
            (
                q1_overall_sparse_df[
                    "candidate_id"
                ]
                == candidate_id
            )
        ]
    )

    assert (
        len(
            row
        )
        == 1
    )

    return row.iloc[
        0
    ]


def aggregate_q2_variant(
    branch,
    candidate_id,
):

    group = (
        q2_by_stratum_df[
            (
                q2_by_stratum_df[
                    "branch"
                ]
                == branch
            )
            &
            (
                q2_by_stratum_df[
                    "candidate_id"
                ]
                == candidate_id
            )
        ]
    )

    assert (
        len(
            group
        )
        == 3
    )

    total_pred = int(
        group[
            "total_pred"
        ].sum()
    )

    total_fp = int(
        group[
            "total_fp"
        ].sum()
    )

    remote = int(
        group[
            "pred_beyond_2R"
        ].sum()
    )

    local_surplus = int(
        group[
            "local_surplus_within_1R"
        ].sum()
    )

    total_gt = int(
        group[
            "total_gt"
        ].sum()
    )

    gt_multi = int(
        group[
            "gt_multiple_local"
        ].sum()
    )

    return {
        "total_pred":
            total_pred,

        "remote_rate":
            (
                remote
                /
                total_pred
                if total_pred > 0
                else 0.0
            ),

        "local_surplus_fraction_of_fp":
            (
                local_surplus
                /
                total_fp
                if total_fp > 0
                else 0.0
            ),

        "gt_multiple_local_rate":
            (
                gt_multi
                /
                total_gt
            ),
    }


def aggregate_q3_checkpoint(
    checkpoint_id,
):

    group = (
        q3_fusion_summary_df[
            q3_fusion_summary_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )

    assert (
        len(
            group
        )
        == 3
    )

    total_gt = int(
        group[
            "total_gt"
        ].sum()
    )

    total_d_only = int(
        group[
            "d_only"
        ].sum()
    )

    total_oracle_tp = int(
        group[
            "oracle_union_tp"
        ].sum()
    )

    total_e_tp = int(
        group[
            "e_tp"
        ].sum()
    )

    total_c_tp = int(
        group[
            "c_tp"
        ].sum()
    )

    total_c_fp = int(
        group[
            "c_fp"
        ].sum()
    )

    total_e_fp = int(
        group[
            "e_fp"
        ].sum()
    )

    delta_tp_vs_c = (
        total_e_tp
        -
        total_c_tp
    )

    delta_fp_vs_c = (
        total_e_fp
        -
        total_c_fp
    )

    return {
        "total_gt":
            total_gt,

        "d_only_rate":
            (
                total_d_only
                /
                total_gt
            ),

        "fusion_fraction_oracle_tp":
            (
                total_e_tp
                /
                total_oracle_tp
            ),

        "delta_tp_vs_c":
            delta_tp_vs_c,

        "delta_fp_vs_c":
            delta_fp_vs_c,
    }


def get_q4_group(
    checkpoint_id,
    outcome,
):

    group = (
        q4_df[
            (
                q4_df[
                    "checkpoint_id"
                ]
                == checkpoint_id
            )
            &
            (
                q4_df[
                    "outcome_vs_b0"
                ]
                == outcome
            )
        ]
    )

    assert (
        len(
            group
        )
        > 0
    )

    return group


# ------------------------------------------------------------
# 5.3 Pull key Step-1 evidence
# ------------------------------------------------------------

q1_c1 = get_q1_variant(
    "C",
    "C1",
)

q1_c2b = get_q1_variant(
    "C",
    "C2b",
)

q1_d6bt = get_q1_variant(
    "D",
    "D6B-T-native",
)

q1_d6b9 = get_q1_variant(
    "D",
    "D6B9-strict",
)


# ------------------------------------------------------------
# 5.4 Pull key Step-2 evidence
# ------------------------------------------------------------

q2_c1 = aggregate_q2_variant(
    "C",
    "C1",
)

q2_c2b = aggregate_q2_variant(
    "C",
    "C2b",
)

q2_d6bt = aggregate_q2_variant(
    "D",
    "D6B-T-native",
)

q2_d6b9 = aggregate_q2_variant(
    "D",
    "D6B9-strict",
)


# ------------------------------------------------------------
# 5.5 Pull key Step-3 evidence
# ------------------------------------------------------------

q3_overall = aggregate_q3_checkpoint(
    "overall_joint"
)

q3_sparse = aggregate_q3_checkpoint(
    "sparse_joint"
)

q3_dense = aggregate_q3_checkpoint(
    "dense_recovery"
)


# ------------------------------------------------------------
# 5.6 Pull key Step-4 evidence
# ------------------------------------------------------------

q4_overall_improved = get_q4_group(
    "overall_joint",
    "improved",
)

q4_overall_worsened = get_q4_group(
    "overall_joint",
    "worsened",
)

q4_sparse_improved = get_q4_group(
    "sparse_joint",
    "improved",
)

q4_sparse_worsened = get_q4_group(
    "sparse_joint",
    "worsened",
)

q4_dense_improved = get_q4_group(
    "dense_recovery",
    "improved",
)

q4_dense_worsened = get_q4_group(
    "dense_recovery",
    "worsened",
)


# ------------------------------------------------------------
# 5.7 Diagnostic evidence table
#
# Important:
# "Supported conclusion" = directly supported by frozen data.
# "Working hypothesis"   = interpretation to test next.
# ------------------------------------------------------------

evidence_records = []


# --------------------------------------------------------
# Finding 1
# --------------------------------------------------------

evidence_records.append(
    {
        "finding_id":
            "F1",

        "question":
            "Does sparse-scene point inflation arise before SAM2?",

        "evidence":
            (
                "Yes. C/D candidate point sets already show substantial "
                "point burden before Group-E fusion or SAM2. "
                f"Across S1-S3, C2b aggregate Pred/GT="
                f"{q1_c2b['aggregate_pred_gt_ratio']:.3f}, "
                f"D6B-T-native={q1_d6bt['aggregate_pred_gt_ratio']:.3f}, "
                f"while D6B9-strict="
                f"{q1_d6b9['aggregate_pred_gt_ratio']:.3f}."
            ),

        "supported_conclusion":
            (
                "The sparse failure is not created by SAM2. "
                "Potential over-generation already exists in the "
                "semantic-reference-point stage."
            ),

        "working_hypothesis":
            (
                "Semantic evidence should not be converted directly "
                "into count-producing prompts without stronger "
                "selection or validation."
            ),

        "status":
            "SUPPORTED",
    }
)


# --------------------------------------------------------
# Finding 2
# --------------------------------------------------------

evidence_records.append(
    {
        "finding_id":
            "F2",

        "question":
            "Are excess semantic points mainly local duplicates?",

        "evidence":
            (
                f"For D6B-T-native across S1-S3, "
                f"{q2_d6bt['remote_rate']:.3f} of predictions are "
                ">2R from every GT point, while only "
                f"{q2_d6bt['local_surplus_fraction_of_fp']:.3f} "
                "of one-to-one FPs are nevertheless within 1R of "
                "some GT."
            ),

        "supported_conclusion":
            (
                "Local duplicate points are not the dominant explanation "
                "for the sparse FP burden. Most excess D6B-T-native "
                "predictions are spatially unsupported relative to "
                "FSC-147 GT points."
            ),

        "working_hypothesis":
            (
                "Simple spatial deduplication alone will not solve "
                "the sparse-scene problem; semantic confidence, "
                "agreement, or proposal validation is likely needed."
            ),

        "status":
            "SUPPORTED",
    }
)


# --------------------------------------------------------
# Finding 3
# --------------------------------------------------------

evidence_records.append(
    {
        "finding_id":
            "F3",

        "question":
            "Do C and D contain genuinely complementary target information?",

        "evidence":
            (
                "Yes. For the frozen C2b + D6B-T-native pair, "
                f"D-only GT coverage across S1-S3 is "
                f"{q3_overall['d_only_rate']:.3f} of all GT objects. "
                "For C1 + D6B9-strict it is "
                f"{q3_dense['d_only_rate']:.3f}."
            ),

        "supported_conclusion":
            (
                "D is not merely redundant noise relative to C. "
                "It recovers a substantial set of GT objects that C misses."
            ),

        "working_hypothesis":
            (
                "Existing C/D evidence should be retained as a research "
                "asset rather than discarded; the main problem is how "
                "to identify useful complementary evidence."
            ),

        "status":
            "SUPPORTED",
    }
)


# --------------------------------------------------------
# Finding 4
# --------------------------------------------------------

evidence_records.append(
    {
        "finding_id":
            "F4",

        "question":
            "Does simple Group-E fusion exploit C/D complementarity efficiently?",

        "evidence":
            (
                f"Across S1-S3, overall_joint recovers "
                f"{q3_overall['fusion_fraction_oracle_tp']:.3f} "
                "of oracle C∪D TP; sparse_joint recovers "
                f"{q3_sparse['fusion_fraction_oracle_tp']:.3f}; "
                f"dense_recovery recovers "
                f"{q3_dense['fusion_fraction_oracle_tp']:.3f}. "
                f"Dense_recovery gains {q3_dense['delta_tp_vs_c']} TP "
                f"relative to C but also changes FP by "
                f"{q3_dense['delta_fp_vs_c']}."
            ),

        "supported_conclusion":
            (
                "The simple fusion families expose a trade-off: "
                "F2 is selective but leaves substantial complementary "
                "coverage unused, whereas F1 preserves much more "
                "complementarity but carries a high FP burden."
            ),

        "working_hypothesis":
            (
                "Structured semantic integration should explicitly model "
                "agreement, complementary recovery, and confidence rather "
                "than using the current simple F1/F2 rules alone."
            ),

        "status":
            "SUPPORTED",
    }
)


# --------------------------------------------------------
# Finding 5
# --------------------------------------------------------

overall_b0_improved_median = float(
    q4_overall_improved[
        "b0_abs_error"
    ].median()
)

overall_b0_worsened_median = float(
    q4_overall_worsened[
        "b0_abs_error"
    ].median()
)

overall_remote_improved_median = float(
    q4_overall_improved[
        "d_remote_rate"
    ].median()
)

overall_remote_worsened_median = float(
    q4_overall_worsened[
        "d_remote_rate"
    ].median()
)

overall_guided_ratio_improved = float(
    q4_overall_improved[
        "guided_pred_gt_ratio"
    ].median()
)

overall_guided_ratio_worsened = float(
    q4_overall_worsened[
        "guided_pred_gt_ratio"
    ].median()
)


evidence_records.append(
    {
        "finding_id":
            "F5",

        "question":
            "When does semantic guidance help rather than hurt B0?",

        "evidence":
            (
                "For overall_joint, improved sparse images have median "
                f"B0 absolute error={overall_b0_improved_median:.3f}, "
                f"versus {overall_b0_worsened_median:.3f} for worsened "
                "images. Median D remote rate is "
                f"{overall_remote_improved_median:.3f} for improved "
                f"versus {overall_remote_worsened_median:.3f} for "
                "worsened images. Median guided Pred/GT is "
                f"{overall_guided_ratio_improved:.3f} versus "
                f"{overall_guided_ratio_worsened:.3f}."
            ),

        "supported_conclusion":
            (
                "On the frozen dev100 sparse subset, semantic guidance "
                "is most useful when B0 severely undercounts and the "
                "semantic evidence is spatially coherent; it often hurts "
                "when B0 is already comparatively close and semantics "
                "injects excessive hypotheses."
            ),

        "working_hypothesis":
            (
                "Semantic evidence may be better used for selective "
                "validation/recovery around an unguided B0 anchor than "
                "as a universal replacement count."
            ),

        "status":
            "SUPPORTED_ON_DEV100",
    }
)


# --------------------------------------------------------
# Finding 6
# --------------------------------------------------------

evidence_records.append(
    {
        "finding_id":
            "F6",

        "question":
            "Is SAM2 currently correcting semantic counting errors?",

        "evidence":
            (
                "No. Under the frozen guided protocol, each retained "
                "semantic reference point produces one retained SAM2 mask, "
                "and final count equals reference-point count."
            ),

        "supported_conclusion":
            (
                "SAM2 currently provides spatial masks but does not "
                "change the guided count. Point-selection errors therefore "
                "propagate directly into counting errors."
            ),

        "working_hypothesis":
            (
                "The semantic-to-SAM2 interface should be reconsidered: "
                "mask verification, consolidation, multi-point prompting, "
                "or B0-mask modification may be more robust than "
                "one-point-one-count."
            ),

        "status":
            "SUPPORTED",
    }
)


q5_evidence_df = pd.DataFrame(
    evidence_records
)


# ------------------------------------------------------------
# 5.8 Architecture decision table
# ------------------------------------------------------------

decision_records = [
    {
        "decision_id":
            "D1",

        "decision_question":
            "Discard existing C/D semantic evidence?",

        "current_answer":
            "NO",

        "basis":
            (
                "Step 3 shows substantial C/D complementarity; "
                "D covers many GT objects unavailable to C."
            ),

        "next_action":
            (
                "Retain existing C/D evidence and investigate "
                "better use before considering complete replacement."
            ),
    },

    {
        "decision_id":
            "D2",

        "decision_question":
            "Continue using simple F1/F2 as the final fusion mechanism?",

        "current_answer":
            "NOT AS FINAL DESIGN",

        "basis":
            (
                "F2 discards much of the available complementary signal; "
                "F1 preserves signal but imports too many excess points."
            ),

        "next_action":
            (
                "Return later to structured semantic integration "
                "(previously deferred Group-E / 13D.3 direction)."
            ),
    },

    {
        "decision_id":
            "D3",

        "decision_question":
            "Continue one semantic point -> one SAM2 mask -> one count?",

        "current_answer":
            "RECONSIDER",

        "basis":
            (
                "The frozen protocol makes final count identical to "
                "semantic-point count, so SAM2 cannot correct upstream "
                "semantic over-generation."
            ),

        "next_action":
            (
                "Evaluate alternative semantic-to-SAM2 interfaces, "
                "including validation/filtering/recovery of B0 masks "
                "and stronger mask-level consolidation."
            ),
    },

    {
        "decision_id":
            "D4",

        "decision_question":
            "Use B0 as a candidate anchor rather than replace it globally?",

        "current_answer":
            "STRONG WORKING HYPOTHESIS",

        "basis":
            (
                "Step 4 shows semantics tends to help severe B0 "
                "undercount cases but often damages images where B0 "
                "is already relatively close."
            ),

        "next_action":
            (
                "Design controlled experiments where semantics modifies, "
                "validates, or recovers B0 rather than replacing it."
            ),
    },

    {
        "decision_id":
            "D5",

        "decision_question":
            "Do we currently need a completely new semantic model/source?",

        "current_answer":
            "NO EVIDENCE YET",

        "basis":
            (
                "Current diagnostics show useful complementary signal "
                "already exists in SAN/CLIP-derived C/D evidence."
            ),

        "next_action":
            (
                "Only revisit semantic-source replacement if improved "
                "selection/integration/interface experiments fail."
            ),
    },
]


q5_decision_df = pd.DataFrame(
    decision_records
)


# ------------------------------------------------------------
# 5.9 Important methodological caveats
# ------------------------------------------------------------

caveats = [
    (
        "All conclusions are developmental results from the frozen "
        "100-image FSC-147 validation subset, not independent "
        "generalization results."
    ),

    (
        "FSC-147 point annotations support point-localization diagnostics "
        "but do not establish instance-mask correctness or mask IoU."
    ),

    (
        "The >2R label means spatially unsupported relative to the "
        "annotation point; it does not prove that every such semantic "
        "response lies on irrelevant background."
    ),

    (
        "Step-4 variables such as GT count, B0 error, D remote rate "
        "relative to GT, and D-only GT coverage use ground truth and "
        "cannot themselves be used for inference-time routing."
    ),

    (
        "The labels sparse_joint and dense_recovery are diagnostic "
        "configuration names only; no GT-derived density routing was "
        "used at inference."
    ),

    (
        "No new model inference, threshold tuning, or test-set access "
        "was performed in this diagnostic notebook."
    ),
]


# ------------------------------------------------------------
# 5.10 Save synthesis
# ------------------------------------------------------------

q5_evidence_df.to_csv(
    Q5_EVIDENCE_FILE,
    index=False,
)

q5_decision_df.to_csv(
    Q5_DECISION_FILE,
    index=False,
)


q5_manifest = {
    "notebook":
        "B1_Sparse_Scene_Failure_Diagnostic",

    "scope":
        {
            "dataset":
                "FSC-147",

            "subset":
                "frozen dev100",

            "primary_analysis":
                "S1-S3",

            "num_sparse_images":
                60,

            "new_inference":
                False,

            "threshold_tuning":
                False,

            "test_set_accessed":
                False,
        },

    "main_supported_findings":
        q5_evidence_df[
            [
                "finding_id",
                "supported_conclusion",
            ]
        ]
        .to_dict(
            orient="records"
        ),

    "architecture_decisions":
        q5_decision_df.to_dict(
            orient="records"
        ),

    "caveats":
        caveats,

    "next_phase_status":
        (
            "Existing semantic evidence retained. "
            "Next experiments should test better semantic selection/"
            "integration and alternative semantic-to-SAM2/B0 interfaces "
            "before considering replacement of the semantic source."
        ),
}


with open(
    Q5_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        q5_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 5.11 Final notebook summary
# ------------------------------------------------------------

print("=" * 125)
print("STEP 5 - DIAGNOSTIC SYNTHESIS / WORKING HYPOTHESES")
print("=" * 125)

print(
    "Diagnostic findings          :",
    len(
        q5_evidence_df
    )
)

print(
    "Architecture decisions       :",
    len(
        q5_decision_df
    )
)

print(
    "New inference                : NO"
)

print(
    "Threshold tuning             : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nSupported diagnostic findings:"
)

display(
    q5_evidence_df[
        [
            "finding_id",
            "question",
            "supported_conclusion",
            "working_hypothesis",
            "status",
        ]
    ]
)


print(
    "\nCurrent architecture decisions:"
)

display(
    q5_decision_df
)


print(
    "\nMethodological caveats:"
)

for i, caveat in enumerate(
    caveats,
    start=1,
):

    print(
        f"{i}. {caveat}"
    )


print(
    "\nSaved evidence summary:",
    Q5_EVIDENCE_FILE,
)

print(
    "Saved architecture decision summary:",
    Q5_DECISION_FILE,
)

print(
    "Saved synthesis manifest:",
    Q5_MANIFEST_FILE,
)

print("=" * 125)

STEP 5 - DIAGNOSTIC SYNTHESIS / WORKING HYPOTHESES
Diagnostic findings          : 6
Architecture decisions       : 5
New inference                : NO
Threshold tuning             : NO
Test set accessed            : NO

Supported diagnostic findings:


,finding_id,question,supported_conclusion,working_hypothesis,status
0,F1,Does sparse-scene point inflation arise before...,The sparse failure is not created by SAM2. Pot...,Semantic evidence should not be converted dire...,SUPPORTED
1,F2,Are excess semantic points mainly local duplic...,Local duplicate points are not the dominant ex...,Simple spatial deduplication alone will not so...,SUPPORTED
2,F3,Do C and D contain genuinely complementary tar...,D is not merely redundant noise relative to C....,Existing C/D evidence should be retained as a ...,SUPPORTED
3,F4,Does simple Group-E fusion exploit C/D complem...,The simple fusion families expose a trade-off:...,Structured semantic integration should explici...,SUPPORTED
4,F5,When does semantic guidance help rather than h...,"On the frozen dev100 sparse subset, semantic g...",Semantic evidence may be better used for selec...,SUPPORTED_ON_DEV100
5,F6,Is SAM2 currently correcting semantic counting...,SAM2 currently provides spatial masks but does...,The semantic-to-SAM2 interface should be recon...,SUPPORTED



Current architecture decisions:


,decision_id,decision_question,current_answer,basis,next_action
0,D1,Discard existing C/D semantic evidence?,NO,Step 3 shows substantial C/D complementarity; ...,Retain existing C/D evidence and investigate b...
1,D2,Continue using simple F1/F2 as the final fusio...,NOT AS FINAL DESIGN,F2 discards much of the available complementar...,Return later to structured semantic integratio...
2,D3,Continue one semantic point -> one SAM2 mask -...,RECONSIDER,The frozen protocol makes final count identica...,Evaluate alternative semantic-to-SAM2 interfac...
3,D4,Use B0 as a candidate anchor rather than repla...,STRONG WORKING HYPOTHESIS,Step 4 shows semantics tends to help severe B0...,Design controlled experiments where semantics ...
4,D5,Do we currently need a completely new semantic...,NO EVIDENCE YET,Current diagnostics show useful complementary ...,Only revisit semantic-source replacement if im...



Methodological caveats:
1. All conclusions are developmental results from the frozen 100-image FSC-147 validation subset, not independent generalization results.
2. FSC-147 point annotations support point-localization diagnostics but do not establish instance-mask correctness or mask IoU.
3. The >2R label means spatially unsupported relative to the annotation point; it does not prove that every such semantic response lies on irrelevant background.
4. Step-4 variables such as GT count, B0 error, D remote rate relative to GT, and D-only GT coverage use ground truth and cannot themselves be used for inference-time routing.
5. The labels sparse_joint and dense_recovery are diagnostic configuration names only; no GT-derived density routing was used at inference.
6. No new model inference, threshold tuning, or test-set access was performed in this diagnostic notebook.

Saved evidence summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Sparse_Scene_Failure_Di